##### HDC PADRAO

In [ ]:
import numpy as np
import os
import re
import time
import io
import gc
import warnings
from scipy.signal import welch, medfilt
from scipy.stats import skew, kurtosis
import antropy
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pyedflib
from collections import defaultdict
from joblib import Parallel, delayed 

warnings.filterwarnings("ignore", category=RuntimeWarning)
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

PATIENTS = [f"chb{str(i).zfill(2)}" for i in range(1, 2)] 
DRIVE_PATH_COMPONENTS = ['TCC EPILEPSIA DATA', 'chb-mit-scalp-eeg-database-1.0.0']
DIMENSIONS, NUM_LEVELS, MAX_CHANNELS, SEED = 10000, 100, 23, 42
WINDOW_SECONDS, OVERLAP_PERCENTAGE = 5.0, 0.5
SMOOTHING_SECONDS = 5.0
FS_ASSUMED = 256
N_JOBS = -1 

def get_drive_service():
    creds = None; creds_folder = 'credentials'; token_path = os.path.join(creds_folder, 'token.json'); credentials_path = os.path.join(creds_folder, 'credentials.json'); os.makedirs(creds_folder, exist_ok=True)
    if os.path.exists(token_path): creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token: creds.refresh(Request())
        else:
            if not os.path.exists(credentials_path): raise FileNotFoundError("ERRO CRÍTICO: 'credentials.json' não encontrado.")
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, SCOPES); creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token: token.write(creds.to_json())
    try: service = build('drive', 'v3', credentials=creds); print("Serviço do Google Drive conectado."); return service
    except Exception as e: print(f"Erro ao construir o serviço do Drive: {e}"); return None
def find_folder_id(service, folder_name, parent_id='root'):
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); items = results.get('files', []); return items[0]['id'] if items else None
def find_folder_id_by_path(service, path_components):
    current_parent_id = 'root'
    for folder_name in path_components:
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{current_parent_id}' in parents"; results = service.files().list(q=query, fields="files(id)").execute(); items = results.get('files', [])
        if not items: print(f"Pasta '{folder_name}' não encontrada."); return None
        current_parent_id = items[0]['id']
    print("Caminho do dataset encontrado no Drive!"); return current_parent_id
def get_files_from_drive_folder(service, folder_id):
    query = f"'{folder_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); return {file['name']: file['id'] for file in results.get('files', [])}
def download_file_locally(service, file_id, local_filename):
    max_retries = 2 ; attempt = 0
    if os.path.exists(local_filename):
        try: os.remove(local_filename)
        except OSError as e: print(f"  [AVISO Download] Não removeu antigo {local_filename}: {e}")
    while attempt <= max_retries:
        try:
            request = service.files().get_media(fileId=file_id)
            print(f"    Download de {os.path.basename(local_filename)} (tentativa {attempt+1}/{max_retries+1})...", end=' ')
            start_dl = time.time()
            with io.FileIO(local_filename, 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, request); done = False
                while not done: status, done = downloader.next_chunk()
            print(f"OK ({time.time() - start_dl:.2f}s).")
            return True
        except Exception as e:
            attempt += 1; print(f"\n      [AVISO Download] Tentativa {attempt}/{max_retries+1} falhou. Erro: [{type(e).__name__}] {e}")
            if os.path.exists(local_filename):
                try: os.remove(local_filename); print("      Arquivo local corrompido removido.")
                except OSError as remove_error: print(f"      AVISO: Não removeu {local_filename}: {remove_error}")
            if attempt > max_retries: print(f"  [ERRO Download] Download falhou."); return False
            print("      Aguardando 3s..."); time.sleep(3)
    return False
def parse_summary_file(file_path):
    seizure_info_final = {}
    try:
        with open(file_path, 'r', errors='ignore') as f: content = f.read()
    except Exception as e: print(f"  [ERRO] Leitura summary: {file_path} - {e}"); return seizure_info_final
    file_blocks = re.split(r'File Name:\s*', content)
    start_pattern = re.compile(r"Seizure\s*\d*\s*Start Time:\s*(\d+)\s*seconds"); end_pattern = re.compile(r"Seizure\s*\d*\s*End Time:\s*(\d+)\s*seconds")
    for block in file_blocks:
        if not block.strip(): continue
        lines = block.strip().split('\n');
        if not lines: continue
        file_name = lines[0].strip(); current_seizures = []
        for i, line in enumerate(lines):
            start_match = start_pattern.search(line)
            if start_match:
                 start_time = int(start_match.group(1)); end_time = None
                 for k in range(i, min(i + 5, len(lines))):
                      end_match_search = end_pattern.search(lines[k])
                      if end_match_search:
                           seizure_num_start = re.search(r"Seizure\s*(\d*)", line); num_s = seizure_num_start.group(1).strip() if seizure_num_start else ''
                           seizure_num_end = re.search(r"Seizure\s*(\d*)", lines[k]); num_e = seizure_num_end.group(1).strip() if seizure_num_end else ''
                           if num_s == num_e or num_s == '' or num_e == '':
                                end_time = int(end_match_search.group(1))
                                if start_time < end_time: current_seizures.append({'start': start_time, 'end': end_time})
                                else: print(f"  AVISO parse_summary: Ignorando {file_name} crise inválida (start >= end): {start_time} >= {end_time}")
                                break
        if current_seizures:
            seizure_info_final[file_name] = [{'interval': (s['start'], s['end']), 'id': f"{file_name}_s{idx+1}"} for idx, s in enumerate(current_seizures)]
    return seizure_info_final

def extract_single_feature_vector(eeg_window, fs=FS_ASSUMED):
    nperseg = len(eeg_window) if len(eeg_window) > 0 else 1
    try: freqs, psd = welch(eeg_window, fs=fs, nperseg=nperseg)
    except ValueError: psd = np.zeros(nperseg // 2 + 1); freqs = np.linspace(0, fs/2, len(psd))
    total_power = np.sum(psd)
    def get_band_power(f_low, f_high): return np.sum(psd[np.logical_and(freqs >= f_low, freqs <= f_high)])
    delta, theta, alpha, beta, gamma = get_band_power(0.5, 4), get_band_power(4, 8), get_band_power(8, 13), get_band_power(13, 30), get_band_power(30, 80)
    band_powers = [p / total_power if total_power > 0 else 0 for p in [delta, theta, alpha, beta, gamma]]
    ratios = [beta / alpha if alpha > 0 else 0, (delta + theta) / (alpha + beta) if (alpha + beta) > 0 else 0]
    try: pe = antropy.perm_entropy(eeg_window, normalize=True)
    except ValueError: pe = 0
    try: se = antropy.spectral_entropy(eeg_window, sf=fs, method='welch', nperseg=nperseg, normalize=True)
    except (ValueError, OSError): se = 0
    try: sae = antropy.sample_entropy(eeg_window)
    except ValueError: sae = 0
    entropies = [pe, se, sae]
    stats = [np.mean(np.abs(eeg_window)), np.std(eeg_window), skew(eeg_window), kurtosis(eeg_window)]
    try: pfd = antropy.petrosian_fd(eeg_window)
    except (ValueError, ZeroDivisionError): pfd = 0
    fractal_dim = pfd; zero_crossings_rate = antropy.num_zerocross(eeg_window) / len(eeg_window) if len(eeg_window)>0 else 0
    features = [total_power] + band_powers + ratios + entropies + stats + [fractal_dim, zero_crossings_rate]
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)

def process_window_wrapper(args):
    window_data, fs = args
    return extract_averaged_features(window_data, fs)

class HDC:
    def __init__(self, dimensions, num_features, num_levels, num_classes=2, seed=None):
        if seed is not None: np.random.seed(seed)
        self.D, self.num_features, self.num_levels, self.num_classes = dimensions, num_features, num_levels, num_classes
        if num_features <= 0: raise ValueError("num_features deve ser > 0")
        self.level_vectors = np.random.choice([-1, 1], size=(num_levels, self.D)); self.feature_vectors = np.random.choice([-1, 1], size=(num_features, self.D))
        self.class_prototypes = np.zeros((self.num_classes, self.D))
    def _quantize(self, data, num_levels):
        min_val, max_val = np.min(data), np.max(data)
        if max_val - min_val < 1e-9: return np.zeros_like(data, dtype=int)
        quantized = np.round((data - min_val) / (max_val - min_val) * (num_levels - 1))
        return np.clip(quantized, 0, num_levels - 1).astype(int)
    def encode(self, x_data):
        num_samples, num_features = x_data.shape
        if num_features != self.num_features: raise ValueError(f"Dimensão errada: esperado {self.num_features}, recebido {num_features}")
        x_quantized = np.array([self._quantize(x_data[:, i], self.num_levels) for i in range(self.num_features)]).T
        encoded_data_sum = np.sum(self.feature_vectors[None, :, :] * self.level_vectors[x_quantized, :], axis=1)
        encoded_data = np.sign(encoded_data_sum); encoded_data[np.all(encoded_data_sum == 0, axis=1)] = 0
        return encoded_data
    def predict(self, x_encoded):
        norm_prototypes = np.sign(self.class_prototypes)
        for k in range(self.num_classes):
             if np.all(norm_prototypes[k] == 0): norm_prototypes[k] = np.random.choice([-1, 1], size=self.D)
        return np.argmax(cosine_similarity(x_encoded, norm_prototypes), axis=1)
    def train_standard(self, x_encoded, y_train):
        self.class_prototypes = np.zeros((self.num_classes, self.D))
        for i in range(self.num_classes):
            class_samples = x_encoded[y_train == i]
            if len(class_samples) > 0: self.class_prototypes[i] = np.sum(class_samples, axis=0)
def post_process_smoothing(predictions, smoothing_seconds=5.0, overlap_percentage=0.5):
    window_step_seconds = WINDOW_SECONDS * (1 - overlap_percentage)
    if window_step_seconds <= 0: smoothing_window_size = 1
    else: smoothing_window_size = int(smoothing_seconds / window_step_seconds)
    if smoothing_window_size < 1: smoothing_window_size = 1
    if smoothing_window_size % 2 == 0: smoothing_window_size += 1
    print(f"Aplicando suavização (janela={smoothing_window_size}, {smoothing_seconds}s)...", end=' ')
    start_smooth = time.time()
    if len(predictions) < smoothing_window_size:
        print(f"\n  AVISO: Predições ({len(predictions)}) < Janela ({smoothing_window_size})...")
        smoothing_window_size = max(1, len(predictions));
        if smoothing_window_size > 0 and smoothing_window_size % 2 == 0: smoothing_window_size = max(1, smoothing_window_size -1)
        if smoothing_window_size == 0 : smoothing_window_size = 1
    try: smoothed_predictions = medfilt(predictions, kernel_size=smoothing_window_size)
    except ValueError as e: print(f"\n  AVISO: Erro medfilt: {e}. Retornando originais."); smoothed_predictions = predictions
    print(f"OK ({time.time() - start_smooth:.2f}s)")
    return smoothed_predictions

def main():
    overall_start_time = time.time()
    all_patient_results = []

    try:
        service = get_drive_service()
        root_id = find_folder_id_by_path(service, DRIVE_PATH_COMPONENTS)
        np.random.seed(SEED)

        for patient_idx, patient_name in enumerate(PATIENTS):
            patient_start_time = time.time()
            print(f"\n{'='*20} Iniciando Paciente {patient_idx+1}/{len(PATIENTS)}: {patient_name} {'='*20}")
            patient_folder_id = find_folder_id(service, patient_name, parent_id=root_id);
            if not patient_folder_id: print(f"  Pasta não encontrada. Pulando."); continue
            files_map = get_files_from_drive_folder(service, patient_folder_id)
            all_edf_files_names = sorted([f for f in files_map.keys() if f.endswith('.edf')])
            summary_filename = f"{patient_name}-summary.txt"
            if summary_filename not in files_map: print(f"  AVISO: {summary_filename} não encontrado. Pulando."); continue
            summary_path = f"./temp_{patient_name}_summary.txt";
            if not download_file_locally(service, files_map[summary_filename], summary_path): continue
            seizure_details = parse_summary_file(summary_path); os.remove(summary_path)
            files_with_seizures_set = set(seizure_details.keys())
            if not files_with_seizures_set: print(f"  AVISO: Nenhuma crise válida no summary. Pulando."); continue

            patient_data_by_seizure = defaultdict(lambda: {'features': [], 'labels': []}); seizure_features_list = []
            nonseizure_features_from_seiz_files = []; nonseizure_features_from_other_files = []; file_processing_times = []

            files_to_process_first = [f for f in all_edf_files_names if f in files_with_seizures_set]
            print(f"  Fase 1a: Processando {len(files_to_process_first)} arquivos COM crises...")
            for i_file, edf_name in enumerate(files_to_process_first):
                file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                if not download_file_locally(service, files_map[edf_name], local_path): continue
                try:
                    with pyedflib.EdfReader(local_path) as r:
                        fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS)
                        signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                        win_samples_file = int(fs_signal * WINDOW_SECONDS)
                        step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE))
                        if step_file <= 0: step_file = 1 
                        current_file_seizures = seizure_details.get(edf_name, [])
                        seizure_intervals_indices = []
                        for seizure_info in current_file_seizures:
                            s, e = seizure_info['interval']
                            s_idx = int(s * fs_signal)
                            e_idx = int(e * fs_signal)
                            seizure_intervals_indices.append({'start_idx': s_idx, 'end_idx': e_idx, 'id': seizure_info['id']})

                        window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file)
                        tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]

                        print(f"    {edf_name}: Extraindo features de {len(tasks)} janelas em paralelo ({N_JOBS} cores)...", end=' ')
                        feature_extraction_start = time.time()
                        all_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks)
                        print(f"OK ({time.time() - feature_extraction_start:.2f}s)")

                        for idx, j in enumerate(window_indices):
                            feats = all_feats[idx] 
                            window_start_idx = j
                            window_end_idx = j + win_samples_file
                            belongs_to_seizure_id = None; is_seiz = False

                            for seiz_indices in seizure_intervals_indices:
                                s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']
                                if max(window_start_idx, s_idx) < min(window_end_idx, e_idx):
                                    is_seiz = True; belongs_to_seizure_id = seiz_indices['id'];
                                    break 

                            if is_seiz:
                                patient_data_by_seizure[belongs_to_seizure_id]['features'].append(feats)
                                patient_data_by_seizure[belongs_to_seizure_id]['labels'].append(1)
                                seizure_features_list.append(feats)
                            else: 
                                too_close = False
                                for seiz_indices in seizure_intervals_indices:
                                    s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']
                                    exclusion_start_idx = s_idx - int(60 * fs_signal) 
                                    exclusion_end_idx = e_idx + int(15 * 60 * fs_signal) 
                                    if max(window_start_idx, exclusion_start_idx) < min(window_end_idx, exclusion_end_idx):
                                        too_close = True; break
                                if not too_close: nonseizure_features_from_seiz_files.append(feats)

                except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                finally:
                    if os.path.exists(local_path): os.remove(local_path)
                gc.collect(); file_end_time = time.time(); file_processing_times.append(file_end_time - file_start_time); avg_time_per_file = np.mean(file_processing_times) if file_processing_times else 0
                remaining_files = len(files_to_process_first) - (i_file + 1); eta_sec = remaining_files * avg_time_per_file
                print(f"    {i_file+1}/{len(files_to_process_first)}: {edf_name} processado em {time.time() - file_start_time:.2f}s. ETA: {eta_sec/60:.1f} min")


            num_seiz_total = len(seizure_features_list)
            if num_seiz_total == 0: print(f"  AVISO: Nenhuma JANELA de crise detectada. Pulando."); continue
            num_nonseiz_found_initial = len(nonseizure_features_from_seiz_files); num_nonseiz_needed_total = num_seiz_total * 10
            print(f"  Fase 1a concluída: {num_seiz_total} janelas de crise."); print(f"                     {num_nonseiz_found_initial} janelas não-crise."); print(f"                     Necessárias {num_nonseiz_needed_total}.")

            if num_nonseiz_found_initial < num_nonseiz_needed_total:
                deficit = num_nonseiz_needed_total - num_nonseiz_found_initial; print(f"  Fase 1b: Buscando mais {deficit} não-crises...")
                files_to_process_later = [f for f in all_edf_files_names if f not in files_with_seizures_set]
                if not files_to_process_later: print("  AVISO: Não há outros arquivos.")
                else:
                    print(f"           Processando até {len(files_to_process_later)} arquivos adicionais (paralelamente)...")
                    nonseiz_collected_later = 0; phase1b_file_times = []

                    for i_file, edf_name in enumerate(files_to_process_later):
                        if nonseiz_collected_later >= deficit: break
                        file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                        if not download_file_locally(service, files_map[edf_name], local_path): continue
                        try:
                            with pyedflib.EdfReader(local_path) as r:
                                fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS)
                                signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                                win_samples_file, step_file = (win_samples, step) if fs_signal == FS_ASSUMED else (int(fs_signal * WINDOW_SECONDS), int(fs_signal * WINDOW_SECONDS * (1.0 - OVERLAP_PERCENTAGE)))
                                if step_file <= 0: step_file = 1

                                window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file)
                                tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                                print(f"      {edf_name}: Extraindo features de {len(tasks)} janelas (paralelo)...", end=' ')
                                feature_extraction_start = time.time()
                                other_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks)
                                print(f"OK ({time.time() - feature_extraction_start:.2f}s)")

                                needed_now = deficit - nonseiz_collected_later
                                added_now = min(needed_now, len(other_feats))
                                nonseizure_features_from_other_files.extend(other_feats[:added_now])
                                nonseiz_collected_later += added_now

                        except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                        finally:
                            if os.path.exists(local_path): os.remove(local_path)
                        gc.collect(); file_end_time = time.time(); phase1b_file_times.append(file_end_time - file_start_time); avg_time_b = np.mean(phase1b_file_times) if phase1b_file_times else 0; remaining_files_b = len(files_to_process_later) - (i_file + 1)
                        files_really_needed = np.ceil(deficit / (nonseiz_collected_later / (i_file + 1))) if (i_file+1)>0 and nonseiz_collected_later>0 else remaining_files_b
                        eta_sec_b = min(remaining_files_b, max(0, files_really_needed - (i_file+1))) * avg_time_b
                        print(f"    {i_file+1}/{len(files_to_process_later)}: {edf_name} em {time.time() - file_start_time:.2f}s ({nonseiz_collected_later}/{deficit}). ETA: {eta_sec_b/60:.1f} min")
                    print(f"  Fase 1b concluída: {nonseiz_collected_later} janelas adicionais coletadas.")
            else: print("  Não foi necessário Fase 1b.")

            patient_potential_nonseizure_features = nonseizure_features_from_seiz_files + nonseizure_features_from_other_files
            num_nonseiz_available = len(patient_potential_nonseizure_features)
            if num_nonseiz_available == 0: print(f"  AVISO CRÍTICO: Nenhuma janela não-crise válida. Pulando."); continue
            if num_nonseiz_available >= num_nonseiz_needed_total:
                indices = np.random.choice(num_nonseiz_available, num_nonseiz_needed_total, replace=False)
                selected_nonseizure_features = [patient_potential_nonseizure_features[i] for i in indices]
            else: selected_nonseizure_features = patient_potential_nonseizure_features; print(f"  AVISO FINAL: Apenas {len(selected_nonseizure_features)}/{num_nonseiz_needed_total} não-crises.")
            num_nonseiz_final = len(selected_nonseizure_features)
            print(f"  Dataset Paciente: {num_seiz_total} crises, {num_nonseiz_final} não-crises (Rácio ~1:{num_nonseiz_final/num_seiz_total:.1f}).")
            seizure_ids = list(patient_data_by_seizure.keys())
            if not seizure_ids or len(seizure_ids) < 2: print(f"  AVISO: Crises insuficientes ({len(seizure_ids)}) para LOSO. Pulando."); continue
            patient_fold_metrics = []
            print(f"  Fase 2: Iniciando LOSO-CV com {len(seizure_ids)} crises...")
            for k, seizure_id_out in enumerate(seizure_ids):
                fold_start_time = time.time()
                print(f"\n    --- FOLD {k+1}/{len(seizure_ids)} ({seizure_id_out} fora) ---")
                X_train_list, y_train_list = [], []; X_test_list, y_test_list = [], []
                test_seizure_data = patient_data_by_seizure[seizure_id_out]; num_seiz_test = len(test_seizure_data['features'])
                if num_seiz_test == 0: print(f"      AVISO: Crise {seizure_id_out} sem janelas. Pulando."); continue
                X_test_list.extend(test_seizure_data['features']); y_test_list.extend([1] * num_seiz_test)
                num_nonseiz_needed_test = num_seiz_test * 10
                if num_nonseiz_final >= num_nonseiz_needed_test:
                     test_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_test, replace=False)
                     X_test_list.extend([selected_nonseizure_features[i] for i in test_nonseiz_indices]); y_test_list.extend([0] * num_nonseiz_needed_test)
                else: X_test_list.extend(selected_nonseizure_features); y_test_list.extend([0] * num_nonseiz_final)
                total_seiz_train = 0
                for seizure_id_train, data in patient_data_by_seizure.items():
                    if seizure_id_train != seizure_id_out:
                        num_windows_seiz_train = len(data['features'])
                        if num_windows_seiz_train > 0: X_train_list.extend(data['features']); y_train_list.extend([1] * num_windows_seiz_train); total_seiz_train += num_windows_seiz_train
                if total_seiz_train == 0: print(f"      AVISO: Sem crises para treino. Pulando."); continue
                num_nonseiz_needed_train = total_seiz_train * 10
                if num_nonseiz_final >= num_nonseiz_needed_train:
                    train_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_train, replace=False)
                    X_train_list.extend([selected_nonseizure_features[i] for i in train_nonseiz_indices]); y_train_list.extend([0] * num_nonseiz_needed_train)
                else: X_train_list.extend(selected_nonseizure_features); y_train_list.extend([0] * num_nonseiz_final)
                X_train, y_train = np.array(X_train_list), np.array(y_train_list); X_test, y_test = np.array(X_test_list), np.array(y_test_list)
                if X_train.ndim != 2 or X_train.shape[1] == 0: print(f"      AVISO: X_train inválido. Shape={X_train.shape}. Pulando."); continue
                print(f"      Treino: {len(X_train)} ({np.bincount(y_train, minlength=2)}), Teste: {len(X_test)} ({np.bincount(y_test, minlength=2)})")
                if len(np.unique(y_train)) < 2 or len(X_test) == 0: print(f"      AVISO: Dados insuficientes. Pulando."); continue

                scaler = StandardScaler(); X_train_scaled = scaler.fit_transform(X_train); X_test_scaled = scaler.transform(X_test)
                num_features_fold = X_train_scaled.shape[1]
                model = HDC(DIMENSIONS, num_features_fold, NUM_LEVELS, seed=SEED)
                encode_start_time = time.time(); X_train_hd = model.encode(X_train_scaled); X_test_hd = model.encode(X_test_scaled); print(f"      Encoding: {time.time() - encode_start_time:.2f}s")
                train_start_time = time.time(); model.train_standard(X_train_hd, y_train); print(f"      Treino: {time.time() - train_start_time:.2f}s") 
                predict_start_time = time.time(); y_pred_raw = model.predict(X_test_hd); print(f"      Predição: {time.time() - predict_start_time:.2f}s")
                y_pred_processed = post_process_smoothing(y_pred_raw, smoothing_seconds=SMOOTHING_SECONDS, overlap_percentage=OVERLAP_PERCENTAGE)
                acc = accuracy_score(y_test, y_pred_processed); f1 = f1_score(y_test, y_pred_processed, zero_division=0); precision = precision_score(y_test, y_pred_processed, zero_division=0); recall = recall_score(y_test, y_pred_processed, zero_division=0)
                patient_fold_metrics.append({'acc': acc, 'f1': f1, 'precision': precision, 'recall': recall})
                print(f"      Resultados: Acc={acc:.3f}, F1={f1:.3f}, P={precision:.3f}, R={recall:.3f}")
                cm = confusion_matrix(y_test, y_pred_processed, labels=[0, 1]); print(f"      Matriz (TN, FP / FN, TP):\n{cm}")
                gc.collect(); print(f"    Fold {k+1} concluído em {time.time() - fold_start_time:.2f}s")

            if patient_fold_metrics:
                avg_acc_pat = np.mean([m['acc'] for m in patient_fold_metrics]); avg_f1_pat = np.mean([m['f1'] for m in patient_fold_metrics]); avg_precision_pat = np.mean([m['precision'] for m in patient_fold_metrics]); avg_recall_pat = np.mean([m['recall'] for m in patient_fold_metrics])
                print(f"\n  Resultado Médio {patient_name}: Acc={avg_acc_pat:.3f}, F1={avg_f1_pat:.3f}, P={avg_precision_pat:.3f}, R={avg_recall_pat:.3f}")
                all_patient_results.append({'acc': avg_acc_pat, 'f1': avg_f1_pat, 'precision': avg_precision_pat, 'recall': avg_recall_pat})
            else: print(f"  Nenhum fold LOSO concluído para {patient_name}.")
            patient_end_time = time.time(); print(f"  Tempo Paciente {patient_name}: {(patient_end_time - patient_start_time) / 60:.2f} min"); gc.collect()

        if all_patient_results:
            print("\n\n" + "="*40 + "\n" + "RESULTADO FINAL (MÉDIA ENTRE PACIENTES)".center(40) + "\n" + "="*40)
            avg_acc_all = np.mean([m['acc'] for m in all_patient_results]); std_acc_all = np.std([m['acc'] for m in all_patient_results])
            avg_f1_all = np.mean([m['f1'] for m in all_patient_results]); std_f1_all = np.std([m['f1'] for m in all_patient_results])
            avg_precision_all = np.mean([m['precision'] for m in all_patient_results]); std_precision_all = np.std([m['precision'] for m in all_patient_results])
            avg_recall_all = np.mean([m['recall'] for m in all_patient_results]); std_recall_all = np.std([m['recall'] for m in all_patient_results])
            print(f"Acurácia Média Geral:      {avg_acc_all:.3f} +/- {std_acc_all:.3f}")
            print(f"F1 Score Médio Geral:      {avg_f1_all:.3f} +/- {std_f1_all:.3f}")
            print(f"Precisão Média Geral:      {avg_precision_all:.3f} +/- {std_precision_all:.3f}")
            print(f"Recall (Sens.) Médio Geral: {avg_recall_all:.3f} +/- {std_recall_all:.3f}")
        else: print("\nNenhum resultado de paciente para média geral.")

    except Exception as e: print(f"\nERRO GERAL: {e}")
    finally:
        overall_end_time = time.time(); print(f"\nTempo total: {(overall_end_time - overall_start_time) / 60:.2f} min", flush=True)

if __name__ == '__main__':
    main()

Serviço do Google Drive conectado.
Caminho do dataset encontrado no Drive!

==================== Iniciando Paciente 1/1: chb01 ====================
    Download de temp_chb01_summary.txt (tentativa 1/3)... OK (2.64s).
  Fase 1a: Processando 7 arquivos COM crises...
    Download de temp_chb01_03.edf (tentativa 1/3)... OK (15.42s).
    chb01_03.edf: Extraindo features de 1439 janelas em paralelo (-1 cores)... OK (122.48s)
    1/7: chb01_03.edf processado em 140.48s. ETA: 14.0 min
    Download de temp_chb01_04.edf (tentativa 1/3)... OK (36.97s).
    chb01_04.edf: Extraindo features de 1439 janelas em paralelo (-1 cores)... OK (101.74s)
    2/7: chb01_04.edf processado em 142.61s. ETA: 11.8 min
    Download de temp_chb01_15.edf (tentativa 1/3)... OK (38.87s).
    chb01_15.edf: Extraindo features de 1439 janelas em paralelo (-1 cores)... OK (102.82s)
    3/7: chb01_15.edf processado em 144.32s. ETA: 9.5 min
    Download de temp_chb01_16.edf (tentativa 1/3)... OK (39.71s).
    chb01_16.edf: 

##### HDC MULTI-PASS

In [ ]:
import numpy as np
import os
import re
import time
import io
import gc
import warnings
from scipy.signal import welch, medfilt
from scipy.stats import skew, kurtosis
import antropy
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pyedflib
from collections import defaultdict
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=RuntimeWarning)
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

PATIENTS = [f"chb{str(i).zfill(2)}" for i in range(1, 2)] 
DRIVE_PATH_COMPONENTS = ['TCC EPILEPSIA DATA', 'chb-mit-scalp-eeg-database-1.0.0']
DIMENSIONS, NUM_LEVELS, MAX_CHANNELS, SEED = 10000, 100, 23, 42
WINDOW_SECONDS, OVERLAP_PERCENTAGE = 5.0, 0.5
SMOOTHING_SECONDS = 5.0
FS_ASSUMED = 256
N_JOBS = -1

EPOCHS = 15
LEARNING_RATE = 0.01
SUBTRACT_WRONG = True 

def get_drive_service():
    creds = None; creds_folder = 'credentials'; token_path = os.path.join(creds_folder, 'token.json'); credentials_path = os.path.join(creds_folder, 'credentials.json'); os.makedirs(creds_folder, exist_ok=True)
    if os.path.exists(token_path): creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token: creds.refresh(Request())
        else:
            if not os.path.exists(credentials_path): raise FileNotFoundError("ERRO CRÍTICO: 'credentials.json' não encontrado.")
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, SCOPES); creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token: token.write(creds.to_json())
    try: service = build('drive', 'v3', credentials=creds); print("Serviço do Google Drive conectado."); return service
    except Exception as e: print(f"Erro ao construir o serviço do Drive: {e}"); return None
def find_folder_id(service, folder_name, parent_id='root'):
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); items = results.get('files', []); return items[0]['id'] if items else None
def find_folder_id_by_path(service, path_components):
    current_parent_id = 'root'
    for folder_name in path_components:
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{current_parent_id}' in parents"; results = service.files().list(q=query, fields="files(id)").execute(); items = results.get('files', [])
        if not items: print(f"Pasta '{folder_name}' não encontrada."); return None
        current_parent_id = items[0]['id']
    print("Caminho do dataset encontrado no Drive!"); return current_parent_id
def get_files_from_drive_folder(service, folder_id):
    query = f"'{folder_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); return {file['name']: file['id'] for file in results.get('files', [])}
def download_file_locally(service, file_id, local_filename):
    max_retries = 2 ; attempt = 0
    if os.path.exists(local_filename):
        try: os.remove(local_filename)
        except OSError as e: print(f"  [AVISO Download] Não removeu antigo {local_filename}: {e}")
    while attempt <= max_retries:
        try:
            request = service.files().get_media(fileId=file_id)
            print(f"    Download de {os.path.basename(local_filename)} (tentativa {attempt+1}/{max_retries+1})...", end=' ')
            start_dl = time.time()
            with io.FileIO(local_filename, 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, request); done = False
                while not done: status, done = downloader.next_chunk()
            print(f"OK ({time.time() - start_dl:.2f}s).")
            return True
        except Exception as e:
            attempt += 1; print(f"\n      [AVISO Download] Tentativa {attempt}/{max_retries+1} falhou. Erro: [{type(e).__name__}] {e}")
            if os.path.exists(local_filename):
                try: os.remove(local_filename); print("      Arquivo local corrompido removido.")
                except OSError as remove_error: print(f"      AVISO: Não removeu {local_filename}: {remove_error}")
            if attempt > max_retries: print(f"  [ERRO Download] Download falhou."); return False
            print("      Aguardando 3s..."); time.sleep(3)
    return False
def parse_summary_file(file_path):
    seizure_info_final = {}
    try:
        with open(file_path, 'r', errors='ignore') as f: content = f.read()
    except Exception as e: print(f"  [ERRO] Leitura summary: {file_path} - {e}"); return seizure_info_final
    file_blocks = re.split(r'File Name:\s*', content)
    start_pattern = re.compile(r"Seizure\s*\d*\s*Start Time:\s*(\d+)\s*seconds"); end_pattern = re.compile(r"Seizure\s*\d*\s*End Time:\s*(\d+)\s*seconds")
    for block in file_blocks:
        if not block.strip(): continue
        lines = block.strip().split('\n');
        if not lines: continue
        file_name = lines[0].strip(); current_seizures = []
        for i, line in enumerate(lines):
            start_match = start_pattern.search(line)
            if start_match:
                 start_time = int(start_match.group(1)); end_time = None
                 for k in range(i, min(i + 5, len(lines))):
                      end_match_search = end_pattern.search(lines[k])
                      if end_match_search:
                           seizure_num_start = re.search(r"Seizure\s*(\d*)", line); num_s = seizure_num_start.group(1).strip() if seizure_num_start else ''
                           seizure_num_end = re.search(r"Seizure\s*(\d*)", lines[k]); num_e = seizure_num_end.group(1).strip() if seizure_num_end else ''
                           if num_s == num_e or num_s == '' or num_e == '':
                                end_time = int(end_match_search.group(1))
                                if start_time < end_time: current_seizures.append({'start': start_time, 'end': end_time})
                                else: print(f"  AVISO parse_summary: Ignorando {file_name} crise inválida (start >= end): {start_time} >= {end_time}")
                                break
        if current_seizures:
            seizure_info_final[file_name] = [{'interval': (s['start'], s['end']), 'id': f"{file_name}_s{idx+1}"} for idx, s in enumerate(current_seizures)]
    return seizure_info_final
def extract_single_feature_vector(eeg_window, fs=FS_ASSUMED):
    nperseg = len(eeg_window) if len(eeg_window) > 0 else 1
    try: freqs, psd = welch(eeg_window, fs=fs, nperseg=nperseg)
    except ValueError: psd = np.zeros(nperseg // 2 + 1); freqs = np.linspace(0, fs/2, len(psd))
    total_power = np.sum(psd)
    def get_band_power(f_low, f_high): return np.sum(psd[np.logical_and(freqs >= f_low, freqs <= f_high)])
    delta, theta, alpha, beta, gamma = get_band_power(0.5, 4), get_band_power(4, 8), get_band_power(8, 13), get_band_power(13, 30), get_band_power(30, 80)
    band_powers = [p / total_power if total_power > 0 else 0 for p in [delta, theta, alpha, beta, gamma]]
    ratios = [beta / alpha if alpha > 0 else 0, (delta + theta) / (alpha + beta) if (alpha + beta) > 0 else 0]
    try: pe = antropy.perm_entropy(eeg_window, normalize=True)
    except ValueError: pe = 0
    try: se = antropy.spectral_entropy(eeg_window, sf=fs, method='welch', nperseg=nperseg, normalize=True)
    except (ValueError, OSError): se = 0
    try: sae = antropy.sample_entropy(eeg_window)
    except ValueError: sae = 0
    entropies = [pe, se, sae]
    stats = [np.mean(np.abs(eeg_window)), np.std(eeg_window), skew(eeg_window), kurtosis(eeg_window)]
    try: pfd = antropy.petrosian_fd(eeg_window)
    except (ValueError, ZeroDivisionError): pfd = 0
    fractal_dim = pfd; zero_crossings_rate = antropy.num_zerocross(eeg_window) / len(eeg_window) if len(eeg_window)>0 else 0
    features = [total_power] + band_powers + ratios + entropies + stats + [fractal_dim, zero_crossings_rate]
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
def extract_averaged_features(window_data, fs=FS_ASSUMED):
     n_channels = window_data.shape[0]
     if n_channels == 0: return np.zeros(16)
     channel_features = [extract_single_feature_vector(window_data[i, :], fs) for i in range(n_channels)]
     return np.mean(channel_features, axis=0)
def process_window_wrapper(args):
    window_data, fs = args
    return extract_averaged_features(window_data, fs)

class HDC:
    def __init__(self, dimensions, num_features, num_levels, num_classes=2, seed=None):
        if seed is not None: np.random.seed(seed)
        self.D, self.num_features, self.num_levels, self.num_classes = dimensions, num_features, num_levels, num_classes
        if num_features <= 0: raise ValueError("num_features deve ser > 0")
        self.level_vectors = np.random.choice([-1, 1], size=(num_levels, self.D)); self.feature_vectors = np.random.choice([-1, 1], size=(num_features, self.D))
        self.class_prototypes = np.zeros((self.num_classes, self.D))
    def _quantize(self, data, num_levels):
        min_val, max_val = np.min(data), np.max(data)
        if max_val - min_val < 1e-9: return np.zeros_like(data, dtype=int)
        quantized = np.round((data - min_val) / (max_val - min_val) * (num_levels - 1))
        return np.clip(quantized, 0, num_levels - 1).astype(int)
    def encode(self, x_data):
        num_samples, num_features = x_data.shape
        if num_features != self.num_features: raise ValueError(f"Dimensão errada: esperado {self.num_features}, recebido {num_features}")
        x_quantized = np.array([self._quantize(x_data[:, i], self.num_levels) for i in range(self.num_features)]).T
        encoded_data_sum = np.sum(self.feature_vectors[None, :, :] * self.level_vectors[x_quantized, :], axis=1)
        encoded_data = np.sign(encoded_data_sum); encoded_data[np.all(encoded_data_sum == 0, axis=1)] = 0
        return encoded_data
    def predict(self, x_encoded):
        norm_prototypes = np.sign(self.class_prototypes)
        for k in range(self.num_classes):
             if np.all(norm_prototypes[k] == 0): norm_prototypes[k] = np.random.choice([-1, 1], size=self.D)
        return np.argmax(cosine_similarity(x_encoded, norm_prototypes), axis=1)

    def train_standard(self, x_encoded, y_train):
        self.class_prototypes = np.zeros((self.num_classes, self.D))
        for i in range(self.num_classes):
            class_samples = x_encoded[y_train == i]
            if len(class_samples) > 0: self.class_prototypes[i] = np.sum(class_samples, axis=0)

    def train_multipass(self, x_encoded, y_train, epochs, lr, subtract_wrong=True):
        self.train_standard(x_encoded, y_train) 
        initial_prototypes = self.class_prototypes.copy()
        print(f"Iniciando Multi-Pass (epochs={epochs}, lr={lr}, subtract={subtract_wrong})...", end=' ')
        start_mp = time.time()

        for epoch in range(epochs):
            current_prototypes_norm = np.sign(self.class_prototypes)
            for k in range(self.num_classes):
                 if np.all(current_prototypes_norm[k] == 0):
                      if np.any(initial_prototypes[k]): current_prototypes_norm[k] = np.sign(initial_prototypes[k])
                      else: current_prototypes_norm[k] = np.random.choice([-1, 1], size=self.D)

            y_pred = np.argmax(cosine_similarity(x_encoded, current_prototypes_norm), axis=1)
            errors = np.sum(y_pred != y_train)

            if errors == 0: print(f"Convergência em {epoch+1} épocas. ", end=''); break

            corrections = np.zeros_like(self.class_prototypes)
            misclassified_idx = np.where(y_pred != y_train)[0]
            correct_labels_mc = y_train[misclassified_idx]
            incorrect_labels_mc = y_pred[misclassified_idx]
            samples_mc = x_encoded[misclassified_idx]

            np.add.at(corrections, correct_labels_mc, lr * samples_mc)
            if subtract_wrong:
                np.subtract.at(corrections, incorrect_labels_mc, lr * samples_mc)

            self.class_prototypes += corrections

        self.class_prototypes = np.sign(self.class_prototypes) 
        print(f"OK ({time.time() - start_mp:.2f}s)")

def post_process_smoothing(predictions, smoothing_seconds=5.0, overlap_percentage=0.5):
    window_step_seconds = WINDOW_SECONDS * (1 - overlap_percentage)
    if window_step_seconds <= 0: smoothing_window_size = 1
    else: smoothing_window_size = int(smoothing_seconds / window_step_seconds)
    if smoothing_window_size < 1: smoothing_window_size = 1
    if smoothing_window_size % 2 == 0: smoothing_window_size += 1
    print(f"Aplicando suavização (janela={smoothing_window_size}, {smoothing_seconds}s)...", end=' ')
    start_smooth = time.time()
    if len(predictions) < smoothing_window_size:
        print(f"\n  AVISO: Predições ({len(predictions)}) < Janela ({smoothing_window_size})...")
        smoothing_window_size = max(1, len(predictions));
        if smoothing_window_size > 0 and smoothing_window_size % 2 == 0: smoothing_window_size = max(1, smoothing_window_size -1)
        if smoothing_window_size == 0 : smoothing_window_size = 1
    try: smoothed_predictions = medfilt(predictions, kernel_size=smoothing_window_size)
    except ValueError as e: print(f"\n  AVISO: Erro medfilt: {e}. Retornando originais."); smoothed_predictions = predictions
    print(f"OK ({time.time() - start_smooth:.2f}s)")
    return smoothed_predictions

def main():
    overall_start_time = time.time()
    all_patient_results = []

    try:
        service = get_drive_service()
        root_id = find_folder_id_by_path(service, DRIVE_PATH_COMPONENTS)
        np.random.seed(SEED)

        for patient_idx, patient_name in enumerate(PATIENTS):
            patient_start_time = time.time()
            print(f"\n{'='*20} Iniciando Paciente {patient_idx+1}/{len(PATIENTS)}: {patient_name} {'='*20}")
            patient_folder_id = find_folder_id(service, patient_name, parent_id=root_id);
            if not patient_folder_id: print(f"  Pasta não encontrada. Pulando."); continue
            files_map = get_files_from_drive_folder(service, patient_folder_id)
            all_edf_files_names = sorted([f for f in files_map.keys() if f.endswith('.edf')])
            summary_filename = f"{patient_name}-summary.txt"
            if summary_filename not in files_map: print(f"  AVISO: {summary_filename} não encontrado. Pulando."); continue
            summary_path = f"./temp_{patient_name}_summary.txt";
            if not download_file_locally(service, files_map[summary_filename], summary_path): continue
            seizure_details = parse_summary_file(summary_path); os.remove(summary_path)
            files_with_seizures_set = set(seizure_details.keys())
            if not files_with_seizures_set: print(f"  AVISO: Nenhuma crise válida no summary. Pulando."); continue
            patient_data_by_seizure = defaultdict(lambda: {'features': [], 'labels': []}); seizure_features_list = []
            nonseizure_features_from_seiz_files = []; nonseizure_features_from_other_files = []; file_processing_times = []
            files_to_process_first = [f for f in all_edf_files_names if f in files_with_seizures_set]
            print(f"  Fase 1a: Processando {len(files_to_process_first)} arquivos COM crises...")
            for i_file, edf_name in enumerate(files_to_process_first):
                file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                if not download_file_locally(service, files_map[edf_name], local_path): continue
                try:
                    with pyedflib.EdfReader(local_path) as r:
                        fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                        win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                        current_file_seizures = seizure_details.get(edf_name, [])
                        seizure_intervals_indices = [{'start_idx': int(s['interval'][0] * fs_signal), 'end_idx': int(s['interval'][1] * fs_signal), 'id': s['id']} for s in current_file_seizures]
                        window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file)
                        tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                        print(f"    {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                        feature_extraction_start = time.time(); all_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                        for idx, j in enumerate(window_indices):
                            feats = all_feats[idx]; window_start_idx = j; window_end_idx = j + win_samples_file
                            belongs_to_seizure_id = None; is_seiz = False
                            for seiz_indices in seizure_intervals_indices:
                                s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']
                                if max(window_start_idx, s_idx) < min(window_end_idx, e_idx): is_seiz = True; belongs_to_seizure_id = seiz_indices['id']; break
                            if is_seiz:
                                patient_data_by_seizure[belongs_to_seizure_id]['features'].append(feats); patient_data_by_seizure[belongs_to_seizure_id]['labels'].append(1); seizure_features_list.append(feats)
                            else:
                                too_close = False
                                for seiz_indices in seizure_intervals_indices:
                                    s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']; exclusion_start_idx = s_idx - int(60 * fs_signal); exclusion_end_idx = e_idx + int(15 * 60 * fs_signal)
                                    if max(window_start_idx, exclusion_start_idx) < min(window_end_idx, exclusion_end_idx): too_close = True; break
                                if not too_close: nonseizure_features_from_seiz_files.append(feats)
                except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                finally:
                    if os.path.exists(local_path): os.remove(local_path)
                gc.collect(); file_end_time = time.time(); file_processing_times.append(file_end_time - file_start_time); avg_time_per_file = np.mean(file_processing_times) if file_processing_times else 0
                remaining_files = len(files_to_process_first) - (i_file + 1); eta_sec = remaining_files * avg_time_per_file
                print(f"    {i_file+1}/{len(files_to_process_first)}: {edf_name} processado em {time.time() - file_start_time:.2f}s. ETA: {eta_sec/60:.1f} min")
            num_seiz_total = len(seizure_features_list)
            if num_seiz_total == 0: print(f"  AVISO: Nenhuma JANELA de crise detectada. Pulando."); continue
            num_nonseiz_found_initial = len(nonseizure_features_from_seiz_files); num_nonseiz_needed_total = num_seiz_total * 10
            print(f"  Fase 1a concluída: {num_seiz_total} janelas crise, {num_nonseiz_found_initial} janelas não-crise."); print(f"                     Necessárias {num_nonseiz_needed_total} não-crise.")
            nonseizure_features_from_other_files = [] 
            if num_nonseiz_found_initial < num_nonseiz_needed_total:
                deficit = num_nonseiz_needed_total - num_nonseiz_found_initial; print(f"  Fase 1b: Buscando mais {deficit} não-crises...")
                files_to_process_later = [f for f in all_edf_files_names if f not in files_with_seizures_set]
                if not files_to_process_later: print("  AVISO: Não há outros arquivos.")
                else:
                    print(f"           Processando até {len(files_to_process_later)} arquivos (paralelo)...")
                    nonseiz_collected_later = 0; phase1b_file_times = []
                    for i_file, edf_name in enumerate(files_to_process_later):
                        if nonseiz_collected_later >= deficit: break
                        file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                        if not download_file_locally(service, files_map[edf_name], local_path): continue
                        try:
                            with pyedflib.EdfReader(local_path) as r:
                                fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                                win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                                window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file); tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                                print(f"      {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                                feature_extraction_start = time.time(); other_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                                needed_now = deficit - nonseiz_collected_later; added_now = min(needed_now, len(other_feats)); nonseizure_features_from_other_files.extend(other_feats[:added_now]); nonseiz_collected_later += added_now
                        except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                        finally:
                            if os.path.exists(local_path): os.remove(local_path)
                        gc.collect(); file_end_time = time.time(); phase1b_file_times.append(file_end_time - file_start_time); avg_time_b = np.mean(phase1b_file_times) if phase1b_file_times else 0; remaining_files_b = len(files_to_process_later) - (i_file + 1)
                        files_really_needed = np.ceil(deficit / (nonseiz_collected_later / (i_file + 1))) if (i_file+1)>0 and nonseiz_collected_later>0 else remaining_files_b
                        eta_sec_b = min(remaining_files_b, max(0, files_really_needed - (i_file+1))) * avg_time_b
                        print(f"    {i_file+1}/{len(files_to_process_later)}: {edf_name} em {time.time() - file_start_time:.2f}s ({nonseiz_collected_later}/{deficit}). ETA: {eta_sec_b/60:.1f} min")
                    print(f"  Fase 1b concluída: {nonseiz_collected_later} janelas adicionais.")
            else: print("  Não foi necessário Fase 1b.")
            patient_potential_nonseizure_features = nonseizure_features_from_seiz_files + nonseizure_features_from_other_files
            num_nonseiz_available = len(patient_potential_nonseizure_features)
            if num_nonseiz_available == 0: print(f"  AVISO CRÍTICO: Nenhuma janela não-crise válida. Pulando."); continue
            if num_nonseiz_available >= num_nonseiz_needed_total:
                indices = np.random.choice(num_nonseiz_available, num_nonseiz_needed_total, replace=False)
                selected_nonseizure_features = [patient_potential_nonseizure_features[i] for i in indices]
            else: selected_nonseizure_features = patient_potential_nonseizure_features; print(f"  AVISO FINAL: Apenas {len(selected_nonseizure_features)}/{num_nonseiz_needed_total} não-crises.")
            num_nonseiz_final = len(selected_nonseizure_features)
            print(f"  Dataset Paciente: {num_seiz_total} crises, {num_nonseiz_final} não-crises (Rácio ~1:{num_nonseiz_final/num_seiz_total:.1f}).")

            seizure_ids = list(patient_data_by_seizure.keys())
            if not seizure_ids or len(seizure_ids) < 2: print(f"  AVISO: Crises insuficientes ({len(seizure_ids)}) para LOSO. Pulando."); continue
            patient_fold_metrics = []
            print(f"  Fase 2: Iniciando LOSO-CV com {len(seizure_ids)} crises...")
            for k, seizure_id_out in enumerate(seizure_ids):
                fold_start_time = time.time()
                print(f"\n    --- FOLD {k+1}/{len(seizure_ids)} ({seizure_id_out} fora) ---")
                X_train_list, y_train_list = [], []; X_test_list, y_test_list = [], []
                test_seizure_data = patient_data_by_seizure[seizure_id_out]; num_seiz_test = len(test_seizure_data['features'])
                if num_seiz_test == 0: print(f"      AVISO: Crise {seizure_id_out} sem janelas. Pulando."); continue
                X_test_list.extend(test_seizure_data['features']); y_test_list.extend([1] * num_seiz_test)
                num_nonseiz_needed_test = num_seiz_test * 10
                if num_nonseiz_final >= num_nonseiz_needed_test:
                     test_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_test, replace=False)
                     X_test_list.extend([selected_nonseizure_features[i] for i in test_nonseiz_indices]); y_test_list.extend([0] * num_nonseiz_needed_test)
                else: X_test_list.extend(selected_nonseizure_features); y_test_list.extend([0] * num_nonseiz_final)
                total_seiz_train = 0
                for seizure_id_train, data in patient_data_by_seizure.items():
                    if seizure_id_train != seizure_id_out:
                        num_windows_seiz_train = len(data['features'])
                        if num_windows_seiz_train > 0: X_train_list.extend(data['features']); y_train_list.extend([1] * num_windows_seiz_train); total_seiz_train += num_windows_seiz_train
                if total_seiz_train == 0: print(f"      AVISO: Sem crises para treino. Pulando."); continue
                num_nonseiz_needed_train = total_seiz_train * 10
                if num_nonseiz_final >= num_nonseiz_needed_train:
                    train_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_train, replace=False)
                    X_train_list.extend([selected_nonseizure_features[i] for i in train_nonseiz_indices]); y_train_list.extend([0] * num_nonseiz_needed_train)
                else: X_train_list.extend(selected_nonseizure_features); y_train_list.extend([0] * num_nonseiz_final)
                X_train, y_train = np.array(X_train_list), np.array(y_train_list); X_test, y_test = np.array(X_test_list), np.array(y_test_list)
                if X_train.ndim != 2 or X_train.shape[1] == 0: print(f"      AVISO: X_train inválido. Shape={X_train.shape}. Pulando."); continue
                print(f"      Treino: {len(X_train)} ({np.bincount(y_train, minlength=2)}), Teste: {len(X_test)} ({np.bincount(y_test, minlength=2)})")
                if len(np.unique(y_train)) < 2 or len(X_test) == 0: print(f"      AVISO: Dados insuficientes. Pulando."); continue

                scaler = StandardScaler(); X_train_scaled = scaler.fit_transform(X_train); X_test_scaled = scaler.transform(X_test)
                num_features_fold = X_train_scaled.shape[1]
                model = HDC(DIMENSIONS, num_features_fold, NUM_LEVELS, seed=SEED)
                encode_start_time = time.time(); X_train_hd = model.encode(X_train_scaled); X_test_hd = model.encode(X_test_scaled); print(f"      Encoding: {time.time() - encode_start_time:.2f}s")

                model.train_multipass(X_train_hd, y_train, epochs=EPOCHS, lr=LEARNING_RATE, subtract_wrong=SUBTRACT_WRONG)

                predict_start_time = time.time(); y_pred_raw = model.predict(X_test_hd); print(f"      Predição: {time.time() - predict_start_time:.2f}s")
                y_pred_processed = post_process_smoothing(y_pred_raw, smoothing_seconds=SMOOTHING_SECONDS, overlap_percentage=OVERLAP_PERCENTAGE)
                acc = accuracy_score(y_test, y_pred_processed); f1 = f1_score(y_test, y_pred_processed, zero_division=0); precision = precision_score(y_test, y_pred_processed, zero_division=0); recall = recall_score(y_test, y_pred_processed, zero_division=0)
                patient_fold_metrics.append({'acc': acc, 'f1': f1, 'precision': precision, 'recall': recall})
                print(f"      Resultados: Acc={acc:.3f}, F1={f1:.3f}, P={precision:.3f}, R={recall:.3f}")
                cm = confusion_matrix(y_test, y_pred_processed, labels=[0, 1]); print(f"      Matriz (TN, FP / FN, TP):\n{cm}")
                gc.collect(); print(f"    Fold {k+1} concluído em {time.time() - fold_start_time:.2f}s")

            if patient_fold_metrics:
                avg_acc_pat = np.mean([m['acc'] for m in patient_fold_metrics]); avg_f1_pat = np.mean([m['f1'] for m in patient_fold_metrics]); avg_precision_pat = np.mean([m['precision'] for m in patient_fold_metrics]); avg_recall_pat = np.mean([m['recall'] for m in patient_fold_metrics])
                print(f"\n  Resultado Médio {patient_name}: Acc={avg_acc_pat:.3f}, F1={avg_f1_pat:.3f}, P={avg_precision_pat:.3f}, R={avg_recall_pat:.3f}")
                all_patient_results.append({'acc': avg_acc_pat, 'f1': avg_f1_pat, 'precision': avg_precision_pat, 'recall': avg_recall_pat})
            else: print(f"  Nenhum fold LOSO concluído para {patient_name}.")
            patient_end_time = time.time(); print(f"  Tempo Paciente {patient_name}: {(patient_end_time - patient_start_time) / 60:.2f} min"); gc.collect()

        if all_patient_results:
            print("\n\n" + "="*40 + "\n" + "RESULTADO FINAL (MÉDIA ENTRE PACIENTES)".center(40) + "\n" + "="*40)
            avg_acc_all = np.mean([m['acc'] for m in all_patient_results]); std_acc_all = np.std([m['acc'] for m in all_patient_results])
            avg_f1_all = np.mean([m['f1'] for m in all_patient_results]); std_f1_all = np.std([m['f1'] for m in all_patient_results])
            avg_precision_all = np.mean([m['precision'] for m in all_patient_results]); std_precision_all = np.std([m['precision'] for m in all_patient_results])
            avg_recall_all = np.mean([m['recall'] for m in all_patient_results]); std_recall_all = np.std([m['recall'] for m in all_patient_results])
            print(f"Acurácia Média Geral:      {avg_acc_all:.3f} +/- {std_acc_all:.3f}")
            print(f"F1 Score Médio Geral:      {avg_f1_all:.3f} +/- {std_f1_all:.3f}")
            print(f"Precisão Média Geral:      {avg_precision_all:.3f} +/- {std_precision_all:.3f}")
            print(f"Recall (Sens.) Médio Geral: {avg_recall_all:.3f} +/- {std_recall_all:.3f}")
        else: print("\nNenhum resultado de paciente para média geral.")

    except Exception as e: print(f"\nERRO GERAL: {e}")
    finally:
        overall_end_time = time.time(); print(f"\nTempo total: {(overall_end_time - overall_start_time) / 60:.2f} min", flush=True)

if __name__ == '__main__':
    main()

Serviço do Google Drive conectado.
Caminho do dataset encontrado no Drive!

==================== Iniciando Paciente 1/1: chb01 ====================
    Download de temp_chb01_summary.txt (tentativa 1/3)... OK (1.23s).
  Fase 1a: Processando 7 arquivos COM crises...
    Download de temp_chb01_03.edf (tentativa 1/3)... OK (40.60s).
    chb01_03.edf: Extraindo features 1439 janelas (-1 cores)... OK (160.66s)
    1/7: chb01_03.edf processado em 205.84s. ETA: 20.6 min
    Download de temp_chb01_04.edf (tentativa 1/3)... OK (48.10s).
    chb01_04.edf: Extraindo features 1439 janelas (-1 cores)... OK (156.59s)
    2/7: chb01_04.edf processado em 208.43s. ETA: 17.3 min
    Download de temp_chb01_15.edf (tentativa 1/3)... OK (44.84s).
    chb01_15.edf: Extraindo features 1439 janelas (-1 cores)... OK (98.30s)
    3/7: chb01_15.edf processado em 145.64s. ETA: 12.4 min
    Download de temp_chb01_16.edf (tentativa 1/3)... OK (43.13s).
    chb01_16.edf: Extraindo features 1439 janelas (-1 cores)...

##### HDC MULTI-CENTROID

In [ ]:
import numpy as np
import os
import re
import time
import io
import gc
import warnings
from scipy.signal import welch, medfilt
from scipy.stats import skew, kurtosis
import antropy
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pyedflib
from collections import defaultdict
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=RuntimeWarning)
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

PATIENTS = [f"chb{str(i).zfill(2)}" for i in range(1, 2)] 
DRIVE_PATH_COMPONENTS = ['TCC EPILEPSIA DATA', 'chb-mit-scalp-eeg-database-1.0.0']
DIMENSIONS, NUM_LEVELS, MAX_CHANNELS, SEED = 10000, 100, 23, 42
WINDOW_SECONDS, OVERLAP_PERCENTAGE = 5.0, 0.5
SMOOTHING_SECONDS = 5.0
FS_ASSUMED = 256
N_JOBS = -1

MC_THRESHOLD = 0.0 
MCr_KEEP_TOP_K = 6 

def get_drive_service():
    creds = None; creds_folder = 'credentials'; token_path = os.path.join(creds_folder, 'token.json'); credentials_path = os.path.join(creds_folder, 'credentials.json'); os.makedirs(creds_folder, exist_ok=True)
    if os.path.exists(token_path): creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token: creds.refresh(Request())
        else:
            if not os.path.exists(credentials_path): raise FileNotFoundError("ERRO CRÍTICO: 'credentials.json' não encontrado.")
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, SCOPES); creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token: token.write(creds.to_json())
    try: service = build('drive', 'v3', credentials=creds); print("Serviço do Google Drive conectado."); return service
    except Exception as e: print(f"Erro ao construir o serviço do Drive: {e}"); return None
def find_folder_id(service, folder_name, parent_id='root'):
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); items = results.get('files', []); return items[0]['id'] if items else None
def find_folder_id_by_path(service, path_components):
    current_parent_id = 'root'
    for folder_name in path_components:
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{current_parent_id}' in parents"; results = service.files().list(q=query, fields="files(id)").execute(); items = results.get('files', [])
        if not items: print(f"Pasta '{folder_name}' não encontrada."); return None
        current_parent_id = items[0]['id']
    print("Caminho do dataset encontrado no Drive!"); return current_parent_id
def get_files_from_drive_folder(service, folder_id):
    query = f"'{folder_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); return {file['name']: file['id'] for file in results.get('files', [])}
def download_file_locally(service, file_id, local_filename):
    max_retries = 2 ; attempt = 0
    if os.path.exists(local_filename):
        try: os.remove(local_filename)
        except OSError as e: print(f"  [AVISO Download] Não removeu antigo {local_filename}: {e}")
    while attempt <= max_retries:
        try:
            request = service.files().get_media(fileId=file_id)
            print(f"    Download de {os.path.basename(local_filename)} (tentativa {attempt+1}/{max_retries+1})...", end=' ')
            start_dl = time.time()
            with io.FileIO(local_filename, 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, request); done = False
                while not done: status, done = downloader.next_chunk()
            print(f"OK ({time.time() - start_dl:.2f}s).")
            return True
        except Exception as e:
            attempt += 1; print(f"\n      [AVISO Download] Tentativa {attempt}/{max_retries+1} falhou. Erro: [{type(e).__name__}] {e}")
            if os.path.exists(local_filename):
                try: os.remove(local_filename); print("      Arquivo local corrompido removido.")
                except OSError as remove_error: print(f"      AVISO: Não removeu {local_filename}: {remove_error}")
            if attempt > max_retries: print(f"  [ERRO Download] Download falhou."); return False
            print("      Aguardando 3s..."); time.sleep(3)
    return False
def parse_summary_file(file_path):
    seizure_info_final = {}
    try:
        with open(file_path, 'r', errors='ignore') as f: content = f.read()
    except Exception as e: print(f"  [ERRO] Leitura summary: {file_path} - {e}"); return seizure_info_final
    file_blocks = re.split(r'File Name:\s*', content)
    start_pattern = re.compile(r"Seizure\s*\d*\s*Start Time:\s*(\d+)\s*seconds"); end_pattern = re.compile(r"Seizure\s*\d*\s*End Time:\s*(\d+)\s*seconds")
    for block in file_blocks:
        if not block.strip(): continue
        lines = block.strip().split('\n');
        if not lines: continue
        file_name = lines[0].strip(); current_seizures = []
        for i, line in enumerate(lines):
            start_match = start_pattern.search(line)
            if start_match:
                 start_time = int(start_match.group(1)); end_time = None
                 for k in range(i, min(i + 5, len(lines))):
                      end_match_search = end_pattern.search(lines[k])
                      if end_match_search:
                           seizure_num_start = re.search(r"Seizure\s*(\d*)", line); num_s = seizure_num_start.group(1).strip() if seizure_num_start else ''
                           seizure_num_end = re.search(r"Seizure\s*(\d*)", lines[k]); num_e = seizure_num_end.group(1).strip() if seizure_num_end else ''
                           if num_s == num_e or num_s == '' or num_e == '':
                                end_time = int(end_match_search.group(1))
                                if start_time < end_time: current_seizures.append({'start': start_time, 'end': end_time})
                                else: print(f"  AVISO parse_summary: Ignorando {file_name} crise inválida (start >= end): {start_time} >= {end_time}")
                                break
        if current_seizures:
            seizure_info_final[file_name] = [{'interval': (s['start'], s['end']), 'id': f"{file_name}_s{idx+1}"} for idx, s in enumerate(current_seizures)]
    return seizure_info_final
def extract_single_feature_vector(eeg_window, fs=FS_ASSUMED):
    nperseg = len(eeg_window) if len(eeg_window) > 0 else 1
    try: freqs, psd = welch(eeg_window, fs=fs, nperseg=nperseg)
    except ValueError: psd = np.zeros(nperseg // 2 + 1); freqs = np.linspace(0, fs/2, len(psd))
    total_power = np.sum(psd)
    def get_band_power(f_low, f_high): return np.sum(psd[np.logical_and(freqs >= f_low, freqs <= f_high)])
    delta, theta, alpha, beta, gamma = get_band_power(0.5, 4), get_band_power(4, 8), get_band_power(8, 13), get_band_power(13, 30), get_band_power(30, 80)
    band_powers = [p / total_power if total_power > 0 else 0 for p in [delta, theta, alpha, beta, gamma]]
    ratios = [beta / alpha if alpha > 0 else 0, (delta + theta) / (alpha + beta) if (alpha + beta) > 0 else 0]
    try: pe = antropy.perm_entropy(eeg_window, normalize=True)
    except ValueError: pe = 0
    try: se = antropy.spectral_entropy(eeg_window, sf=fs, method='welch', nperseg=nperseg, normalize=True)
    except (ValueError, OSError): se = 0
    try: sae = antropy.sample_entropy(eeg_window)
    except ValueError: sae = 0
    entropies = [pe, se, sae]
    stats = [np.mean(np.abs(eeg_window)), np.std(eeg_window), skew(eeg_window), kurtosis(eeg_window)]
    try: pfd = antropy.petrosian_fd(eeg_window)
    except (ValueError, ZeroDivisionError): pfd = 0
    fractal_dim = pfd; zero_crossings_rate = antropy.num_zerocross(eeg_window) / len(eeg_window) if len(eeg_window)>0 else 0
    features = [total_power] + band_powers + ratios + entropies + stats + [fractal_dim, zero_crossings_rate]
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
def extract_averaged_features(window_data, fs=FS_ASSUMED):
     n_channels = window_data.shape[0]
     if n_channels == 0: return np.zeros(16)
     channel_features = [extract_single_feature_vector(window_data[i, :], fs) for i in range(n_channels)]
     return np.mean(channel_features, axis=0)
def process_window_wrapper(args):
    window_data, fs = args
    return extract_averaged_features(window_data, fs)

class HDC:
    def __init__(self, dimensions, num_features, num_levels, num_classes=2, seed=None):
        if seed is not None: np.random.seed(seed)
        self.D, self.num_features, self.num_levels, self.num_classes = dimensions, num_features, num_levels, num_classes
        if num_features <= 0: raise ValueError("num_features deve ser > 0")
        self.level_vectors = np.random.choice([-1, 1], size=(num_levels, self.D)); self.feature_vectors = np.random.choice([-1, 1], size=(num_features, self.D))
        self.class_prototypes = np.zeros((self.num_classes, self.D))
        self._sub_prototypes = [] 
        self._sub_proto_labels = [] 
    def _quantize(self, data, num_levels):
        min_val, max_val = np.min(data), np.max(data)
        if max_val - min_val < 1e-9: return np.zeros_like(data, dtype=int)
        quantized = np.round((data - min_val) / (max_val - min_val) * (num_levels - 1))
        return np.clip(quantized, 0, num_levels - 1).astype(int)
    def encode(self, x_data):
        num_samples, num_features = x_data.shape
        if num_features != self.num_features: raise ValueError(f"Dimensão errada: esperado {self.num_features}, recebido {num_features}")
        x_quantized = np.array([self._quantize(x_data[:, i], self.num_levels) for i in range(self.num_features)]).T
        encoded_data_sum = np.sum(self.feature_vectors[None, :, :] * self.level_vectors[x_quantized, :], axis=1)
        return encoded_data_sum 

    def predict(self, x_encoded):
        norm_prototypes = np.sign(self.class_prototypes)
        for k in range(self.num_classes):
             if np.all(norm_prototypes[k] == 0): norm_prototypes[k] = np.random.choice([-1, 1], size=self.D)
        x_encoded_bin = np.sign(x_encoded); x_encoded_bin[np.all(x_encoded_bin == 0, axis=1)] = 0
        return np.argmax(cosine_similarity(x_encoded_bin, norm_prototypes), axis=1)

    def train_multicentroid(self, x_encoded, y_train, threshold, keep_top_k):
        print(f"Iniciando Multi-Centroid (Thresh={threshold}, K={keep_top_k})...", end=' ')
        start_mc = time.time()
        self._sub_prototypes = [] 
        self._sub_proto_labels = []
        initial_sub_prototypes_accum = [] 
        initial_sub_proto_labels = []
        sample_counts = [] 

        for i in range(len(y_train)):
            sample_hv = x_encoded[i]; correct_label = y_train[i]; wrong_label = 1 - correct_label
            sim_correct = -np.inf; best_correct_idx = -1; sim_wrong = -np.inf

            if initial_sub_prototypes_accum:
                sample_hv_bin = np.sign(sample_hv); sample_hv_bin[sample_hv_bin==0] = 1 
                current_sub_protos_bin = np.sign(np.array(initial_sub_prototypes_accum))
                for row_idx in range(current_sub_protos_bin.shape[0]):
                    if np.all(current_sub_protos_bin[row_idx] == 0):
                        current_sub_protos_bin[row_idx] = np.random.choice([-1, 1], size=self.D)

                similarities = cosine_similarity(sample_hv_bin.reshape(1, -1), current_sub_protos_bin)[0]
                correct_indices = [idx for idx, lbl in enumerate(initial_sub_proto_labels) if lbl == correct_label]
                if correct_indices:
                    sim_correct_local_idx = np.argmax(similarities[correct_indices])
                    sim_correct = similarities[correct_indices][sim_correct_local_idx]
                    best_correct_idx = correct_indices[sim_correct_local_idx]
                wrong_indices = [idx for idx, lbl in enumerate(initial_sub_proto_labels) if lbl == wrong_label]
                if wrong_indices: sim_wrong = np.max(similarities[wrong_indices])

            create_new = (sim_wrong > sim_correct) or (sim_correct == -np.inf)
            if create_new:
                initial_sub_prototypes_accum.append(sample_hv.copy()) 
                initial_sub_proto_labels.append(correct_label)
                sample_counts.append(1) 
            else:
                 initial_sub_prototypes_accum[best_correct_idx] += sample_hv 
                 sample_counts[best_correct_idx] += 1 

        print(f"Fase 1 (MC): {len(initial_sub_prototypes_accum)} subs criados ({np.bincount(initial_sub_proto_labels, minlength=2)}). ", end='')
        if not initial_sub_prototypes_accum: print("AVISO: Nenhum sub criado."); self.class_prototypes = np.zeros((self.num_classes, self.D)); return

        print(f"Fase 2 (MCr): Reduzindo para ~{keep_top_k}/classe... ", end='')
        self._sub_prototypes = [] 
        self._sub_proto_labels = []
        for label in range(self.num_classes):
            indices_of_class = [i for i, lbl in enumerate(initial_sub_proto_labels) if lbl == label]
            if not indices_of_class: continue
            counts_of_class = [sample_counts[i] for i in indices_of_class] 
            num_to_keep = min(keep_top_k, len(indices_of_class))
            top_k_local_indices = np.argsort(counts_of_class)[-num_to_keep:]
            top_k_global_indices = [indices_of_class[i] for i in top_k_local_indices]
            self._sub_prototypes.extend([initial_sub_prototypes_accum[i] for i in top_k_global_indices]) 
            self._sub_proto_labels.extend([label] * num_to_keep)

        if not self._sub_prototypes: print("AVISO: Nenhum sub restou após MCr."); self.class_prototypes = np.zeros((self.num_classes, self.D)); return
        print(f"{len(self._sub_prototypes)} subs mantidos. ", end='')

        final_prototypes = np.zeros((self.num_classes, self.D))
        for label in range(self.num_classes):
            indices = [i for i, l in enumerate(self._sub_proto_labels) if l == label]
            if indices: final_prototypes[label] = np.sum(np.array(self._sub_prototypes)[indices], axis=0)
        self.class_prototypes = np.sign(final_prototypes) 
        print(f"OK ({time.time() - start_mc:.2f}s)")

def post_process_smoothing(predictions, smoothing_seconds=5.0, overlap_percentage=0.5):
    window_step_seconds = WINDOW_SECONDS * (1 - overlap_percentage)
    if window_step_seconds <= 0: smoothing_window_size = 1
    else: smoothing_window_size = int(smoothing_seconds / window_step_seconds)
    if smoothing_window_size < 1: smoothing_window_size = 1
    if smoothing_window_size % 2 == 0: smoothing_window_size += 1
    print(f"Aplicando suavização (janela={smoothing_window_size}, {smoothing_seconds}s)...", end=' ')
    start_smooth = time.time()
    if len(predictions) < smoothing_window_size:
        print(f"\n  AVISO: Predições ({len(predictions)}) < Janela ({smoothing_window_size})...")
        smoothing_window_size = max(1, len(predictions));
        if smoothing_window_size > 0 and smoothing_window_size % 2 == 0: smoothing_window_size = max(1, smoothing_window_size -1)
        if smoothing_window_size == 0 : smoothing_window_size = 1
    try: smoothed_predictions = medfilt(predictions, kernel_size=smoothing_window_size)
    except ValueError as e: print(f"\n  AVISO: Erro medfilt: {e}. Retornando originais."); smoothed_predictions = predictions
    print(f"OK ({time.time() - start_smooth:.2f}s)")
    return smoothed_predictions

def main():
    overall_start_time = time.time()
    all_patient_results = []

    try:
        service = get_drive_service()
        root_id = find_folder_id_by_path(service, DRIVE_PATH_COMPONENTS)
        np.random.seed(SEED)

        for patient_idx, patient_name in enumerate(PATIENTS):
            patient_start_time = time.time()
            print(f"\n{'='*20} Iniciando Paciente {patient_idx+1}/{len(PATIENTS)}: {patient_name} {'='*20}")
            patient_folder_id = find_folder_id(service, patient_name, parent_id=root_id);
            if not patient_folder_id: print(f"  Pasta não encontrada. Pulando."); continue
            files_map = get_files_from_drive_folder(service, patient_folder_id)
            all_edf_files_names = sorted([f for f in files_map.keys() if f.endswith('.edf')])
            summary_filename = f"{patient_name}-summary.txt"
            if summary_filename not in files_map: print(f"  AVISO: {summary_filename} não encontrado. Pulando."); continue
            summary_path = f"./temp_{patient_name}_summary.txt";
            if not download_file_locally(service, files_map[summary_filename], summary_path): continue
            seizure_details = parse_summary_file(summary_path); os.remove(summary_path)
            files_with_seizures_set = set(seizure_details.keys())
            if not files_with_seizures_set: print(f"  AVISO: Nenhuma crise válida no summary. Pulando."); continue
            patient_data_by_seizure = defaultdict(lambda: {'features': [], 'labels': []}); seizure_features_list = []
            nonseizure_features_from_seiz_files = []; nonseizure_features_from_other_files = []; file_processing_times = []
            files_to_process_first = [f for f in all_edf_files_names if f in files_with_seizures_set]
            print(f"  Fase 1a: Processando {len(files_to_process_first)} arquivos COM crises...")
            for i_file, edf_name in enumerate(files_to_process_first):
                file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                if not download_file_locally(service, files_map[edf_name], local_path): continue
                try:
                    with pyedflib.EdfReader(local_path) as r:
                        fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                        win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                        current_file_seizures = seizure_details.get(edf_name, [])
                        seizure_intervals_indices = [{'start_idx': int(s['interval'][0] * fs_signal), 'end_idx': int(s['interval'][1] * fs_signal), 'id': s['id']} for s in current_file_seizures]
                        window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file)
                        tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                        print(f"    {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                        feature_extraction_start = time.time(); all_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                        for idx, j in enumerate(window_indices):
                            feats = all_feats[idx]; window_start_idx = j; window_end_idx = j + win_samples_file
                            belongs_to_seizure_id = None; is_seiz = False
                            for seiz_indices in seizure_intervals_indices:
                                s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']
                                if max(window_start_idx, s_idx) < min(window_end_idx, e_idx): is_seiz = True; belongs_to_seizure_id = seiz_indices['id']; break
                            if is_seiz:
                                patient_data_by_seizure[belongs_to_seizure_id]['features'].append(feats); patient_data_by_seizure[belongs_to_seizure_id]['labels'].append(1); seizure_features_list.append(feats)
                            else:
                                too_close = False
                                for seiz_indices in seizure_intervals_indices:
                                    s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']; exclusion_start_idx = s_idx - int(60 * fs_signal); exclusion_end_idx = e_idx + int(15 * 60 * fs_signal)
                                    if max(window_start_idx, exclusion_start_idx) < min(window_end_idx, exclusion_end_idx): too_close = True; break
                                if not too_close: nonseizure_features_from_seiz_files.append(feats)
                except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                finally:
                    if os.path.exists(local_path): os.remove(local_path)
                gc.collect(); file_end_time = time.time(); file_processing_times.append(file_end_time - file_start_time); avg_time_per_file = np.mean(file_processing_times) if file_processing_times else 0
                remaining_files = len(files_to_process_first) - (i_file + 1); eta_sec = remaining_files * avg_time_per_file
                print(f"    {i_file+1}/{len(files_to_process_first)}: {edf_name} processado em {time.time() - file_start_time:.2f}s. ETA: {eta_sec/60:.1f} min")
            num_seiz_total = len(seizure_features_list)
            if num_seiz_total == 0: print(f"  AVISO: Nenhuma JANELA de crise detectada. Pulando."); continue
            num_nonseiz_found_initial = len(nonseizure_features_from_seiz_files); num_nonseiz_needed_total = num_seiz_total * 10
            print(f"  Fase 1a concluída: {num_seiz_total} janelas crise, {num_nonseiz_found_initial} não-crise."); print(f"                     Necessárias {num_nonseiz_needed_total}.")
            nonseizure_features_from_other_files = [] 
            if num_nonseiz_found_initial < num_nonseiz_needed_total:
                deficit = num_nonseiz_needed_total - num_nonseiz_found_initial; print(f"  Fase 1b: Buscando mais {deficit} não-crises...")
                files_to_process_later = [f for f in all_edf_files_names if f not in files_with_seizures_set]
                if not files_to_process_later: print("  AVISO: Não há outros arquivos.")
                else:
                    print(f"           Processando até {len(files_to_process_later)} arquivos (paralelo)...")
                    nonseiz_collected_later = 0; phase1b_file_times = []
                    for i_file, edf_name in enumerate(files_to_process_later):
                        if nonseiz_collected_later >= deficit: break
                        file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                        if not download_file_locally(service, files_map[edf_name], local_path): continue
                        try:
                            with pyedflib.EdfReader(local_path) as r:
                                fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                                win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                                window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file); tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                                print(f"      {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                                feature_extraction_start = time.time(); other_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                                needed_now = deficit - nonseiz_collected_later; added_now = min(needed_now, len(other_feats)); nonseizure_features_from_other_files.extend(other_feats[:added_now]); nonseiz_collected_later += added_now
                        except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                        finally:
                            if os.path.exists(local_path): os.remove(local_path)
                        gc.collect(); file_end_time = time.time(); phase1b_file_times.append(file_end_time - file_start_time); avg_time_b = np.mean(phase1b_file_times) if phase1b_file_times else 0; remaining_files_b = len(files_to_process_later) - (i_file + 1)
                        files_really_needed = np.ceil(deficit / (nonseiz_collected_later / (i_file + 1))) if (i_file+1)>0 and nonseiz_collected_later>0 else remaining_files_b
                        eta_sec_b = min(remaining_files_b, max(0, files_really_needed - (i_file+1))) * avg_time_b
                        print(f"    {i_file+1}/{len(files_to_process_later)}: {edf_name} em {time.time() - file_start_time:.2f}s ({nonseiz_collected_later}/{deficit}). ETA: {eta_sec_b/60:.1f} min")
                    print(f"  Fase 1b concluída: {nonseiz_collected_later} janelas adicionais.")
            else: print("  Não foi necessário Fase 1b.")
            patient_potential_nonseizure_features = nonseizure_features_from_seiz_files + nonseizure_features_from_other_files
            num_nonseiz_available = len(patient_potential_nonseizure_features)
            if num_nonseiz_available == 0: print(f"  AVISO CRÍTICO: Nenhuma janela não-crise válida. Pulando."); continue
            if num_nonseiz_available >= num_nonseiz_needed_total:
                indices = np.random.choice(num_nonseiz_available, num_nonseiz_needed_total, replace=False)
                selected_nonseizure_features = [patient_potential_nonseizure_features[i] for i in indices]
            else: selected_nonseizure_features = patient_potential_nonseizure_features; print(f"  AVISO FINAL: Apenas {len(selected_nonseizure_features)}/{num_nonseiz_needed_total} não-crises.")
            num_nonseiz_final = len(selected_nonseizure_features)
            print(f"  Dataset Paciente: {num_seiz_total} crises, {num_nonseiz_final} não-crises (Rácio ~1:{num_nonseiz_final/num_seiz_total:.1f}).")

            seizure_ids = list(patient_data_by_seizure.keys())
            if not seizure_ids or len(seizure_ids) < 2: print(f"  AVISO: Crises insuficientes ({len(seizure_ids)}) para LOSO. Pulando."); continue
            patient_fold_metrics = []
            print(f"  Fase 2: Iniciando LOSO-CV com {len(seizure_ids)} crises...")
            for k, seizure_id_out in enumerate(seizure_ids):
                fold_start_time = time.time()
                print(f"\n    --- FOLD {k+1}/{len(seizure_ids)} ({seizure_id_out} fora) ---")
                X_train_list, y_train_list = [], []; X_test_list, y_test_list = [], []
                test_seizure_data = patient_data_by_seizure[seizure_id_out]; num_seiz_test = len(test_seizure_data['features'])
                if num_seiz_test == 0: print(f"      AVISO: Crise {seizure_id_out} sem janelas. Pulando."); continue
                X_test_list.extend(test_seizure_data['features']); y_test_list.extend([1] * num_seiz_test)
                num_nonseiz_needed_test = num_seiz_test * 10
                if num_nonseiz_final >= num_nonseiz_needed_test:
                     test_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_test, replace=False)
                     X_test_list.extend([selected_nonseizure_features[i] for i in test_nonseiz_indices]); y_test_list.extend([0] * num_nonseiz_needed_test)
                else: X_test_list.extend(selected_nonseizure_features); y_test_list.extend([0] * num_nonseiz_final)
                total_seiz_train = 0
                for seizure_id_train, data in patient_data_by_seizure.items():
                    if seizure_id_train != seizure_id_out:
                        num_windows_seiz_train = len(data['features'])
                        if num_windows_seiz_train > 0: X_train_list.extend(data['features']); y_train_list.extend([1] * num_windows_seiz_train); total_seiz_train += num_windows_seiz_train
                if total_seiz_train == 0: print(f"      AVISO: Sem crises para treino. Pulando."); continue
                num_nonseiz_needed_train = total_seiz_train * 10
                if num_nonseiz_final >= num_nonseiz_needed_train:
                    train_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_train, replace=False)
                    X_train_list.extend([selected_nonseizure_features[i] for i in train_nonseiz_indices]); y_train_list.extend([0] * num_nonseiz_needed_train)
                else: X_train_list.extend(selected_nonseizure_features); y_train_list.extend([0] * num_nonseiz_final)
                X_train, y_train = np.array(X_train_list), np.array(y_train_list); X_test, y_test = np.array(X_test_list), np.array(y_test_list)
                if X_train.ndim != 2 or X_train.shape[1] == 0: print(f"      AVISO: X_train inválido. Shape={X_train.shape}. Pulando."); continue
                print(f"      Treino: {len(X_train)} ({np.bincount(y_train, minlength=2)}), Teste: {len(X_test)} ({np.bincount(y_test, minlength=2)})")
                if len(np.unique(y_train)) < 2 or len(X_test) == 0: print(f"      AVISO: Dados insuficientes. Pulando."); continue

                scaler = StandardScaler(); X_train_scaled = scaler.fit_transform(X_train); X_test_scaled = scaler.transform(X_test)
                num_features_fold = X_train_scaled.shape[1]
                model = HDC(DIMENSIONS, num_features_fold, NUM_LEVELS, seed=SEED)
                encode_start_time = time.time();
                X_train_hd = model.encode(X_train_scaled);
                X_test_hd = model.encode(X_test_scaled);
                print(f"      Encoding: {time.time() - encode_start_time:.2f}s")

                model.train_multicentroid(X_train_hd, y_train, threshold=MC_THRESHOLD, keep_top_k=MCr_KEEP_TOP_K)

                predict_start_time = time.time(); y_pred_raw = model.predict(X_test_hd); print(f"      Predição: {time.time() - predict_start_time:.2f}s") 
                y_pred_processed = post_process_smoothing(y_pred_raw, smoothing_seconds=SMOOTHING_SECONDS, overlap_percentage=OVERLAP_PERCENTAGE)
                acc = accuracy_score(y_test, y_pred_processed); f1 = f1_score(y_test, y_pred_processed, zero_division=0); precision = precision_score(y_test, y_pred_processed, zero_division=0); recall = recall_score(y_test, y_pred_processed, zero_division=0)
                patient_fold_metrics.append({'acc': acc, 'f1': f1, 'precision': precision, 'recall': recall})
                print(f"      Resultados: Acc={acc:.3f}, F1={f1:.3f}, P={precision:.3f}, R={recall:.3f}")
                cm = confusion_matrix(y_test, y_pred_processed, labels=[0, 1]); print(f"      Matriz (TN, FP / FN, TP):\n{cm}")
                gc.collect(); print(f"    Fold {k+1} concluído em {time.time() - fold_start_time:.2f}s")

            if patient_fold_metrics:
                avg_acc_pat = np.mean([m['acc'] for m in patient_fold_metrics]); avg_f1_pat = np.mean([m['f1'] for m in patient_fold_metrics]); avg_precision_pat = np.mean([m['precision'] for m in patient_fold_metrics]); avg_recall_pat = np.mean([m['recall'] for m in patient_fold_metrics])
                print(f"\n  Resultado Médio {patient_name}: Acc={avg_acc_pat:.3f}, F1={avg_f1_pat:.3f}, P={avg_precision_pat:.3f}, R={avg_recall_pat:.3f}")
                all_patient_results.append({'acc': avg_acc_pat, 'f1': avg_f1_pat, 'precision': avg_precision_pat, 'recall': avg_recall_pat})
            else: print(f"  Nenhum fold LOSO concluído para {patient_name}.")
            patient_end_time = time.time(); print(f"  Tempo Paciente {patient_name}: {(patient_end_time - patient_start_time) / 60:.2f} min"); gc.collect()

        if all_patient_results:
            print("\n\n" + "="*40 + "\n" + "RESULTADO FINAL (MÉDIA ENTRE PACIENTES)".center(40) + "\n" + "="*40)
            avg_acc_all = np.mean([m['acc'] for m in all_patient_results]); std_acc_all = np.std([m['acc'] for m in all_patient_results])
            avg_f1_all = np.mean([m['f1'] for m in all_patient_results]); std_f1_all = np.std([m['f1'] for m in all_patient_results])
            avg_precision_all = np.mean([m['precision'] for m in all_patient_results]); std_precision_all = np.std([m['precision'] for m in all_patient_results])
            avg_recall_all = np.mean([m['recall'] for m in all_patient_results]); std_recall_all = np.std([m['recall'] for m in all_patient_results])
            print(f"Acurácia Média Geral:      {avg_acc_all:.3f} +/- {std_acc_all:.3f}")
            print(f"F1 Score Médio Geral:      {avg_f1_all:.3f} +/- {std_f1_all:.3f}")
            print(f"Precisão Média Geral:      {avg_precision_all:.3f} +/- {std_precision_all:.3f}")
            print(f"Recall (Sens.) Médio Geral: {avg_recall_all:.3f} +/- {std_recall_all:.3f}")
        else: print("\nNenhum resultado de paciente para média geral.")

    except Exception as e: print(f"\nERRO GERAL: {e}")
    finally:
        overall_end_time = time.time(); print(f"\nTempo total: {(overall_end_time - overall_start_time) / 60:.2f} min", flush=True)

if __name__ == '__main__':
    main()

Serviço do Google Drive conectado.
Caminho do dataset encontrado no Drive!

==================== Iniciando Paciente 1/1: chb01 ====================
    Download de temp_chb01_summary.txt (tentativa 1/3)... OK (1.29s).
  Fase 1a: Processando 7 arquivos COM crises...
    Download de temp_chb01_03.edf (tentativa 1/3)... OK (16.66s).
    chb01_03.edf: Extraindo features 1439 janelas (-1 cores)... OK (100.82s)
    1/7: chb01_03.edf processado em 120.00s. ETA: 12.0 min
    Download de temp_chb01_04.edf (tentativa 1/3)... OK (36.56s).
    chb01_04.edf: Extraindo features 1439 janelas (-1 cores)... OK (99.32s)
    2/7: chb01_04.edf processado em 138.65s. ETA: 10.8 min
    Download de temp_chb01_15.edf (tentativa 1/3)... OK (38.75s).
    chb01_15.edf: Extraindo features 1439 janelas (-1 cores)... OK (100.13s)
    3/7: chb01_15.edf processado em 141.44s. ETA: 8.9 min
    Download de temp_chb01_16.edf (tentativa 1/3)... OK (40.69s).
    chb01_16.edf: Extraindo features 1439 janelas (-1 cores)... 

##### HDC MP + MC

In [ ]:
import numpy as np
import os
import re
import time
import io
import gc
import warnings
from scipy.signal import welch, medfilt
from scipy.stats import skew, kurtosis
import antropy
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pyedflib
from collections import defaultdict
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=RuntimeWarning)
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

PATIENTS = [f"chb{str(i).zfill(2)}" for i in range(1, 2)] 
DRIVE_PATH_COMPONENTS = ['TCC EPILEPSIA DATA', 'chb-mit-scalp-eeg-database-1.0.0']
DIMENSIONS, NUM_LEVELS, MAX_CHANNELS, SEED = 10000, 100, 23, 42
WINDOW_SECONDS, OVERLAP_PERCENTAGE = 5.0, 0.5
SMOOTHING_SECONDS = 5.0
FS_ASSUMED = 256
N_JOBS = -1

MC_THRESHOLD = 0.0 
MCr_KEEP_TOP_K = 6 
EPOCHS = 15 
LEARNING_RATE = 0.01 
SUBTRACT_WRONG = True 

def get_drive_service():
    creds = None; creds_folder = 'credentials'; token_path = os.path.join(creds_folder, 'token.json'); credentials_path = os.path.join(creds_folder, 'credentials.json'); os.makedirs(creds_folder, exist_ok=True)
    if os.path.exists(token_path): creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token: creds.refresh(Request())
        else:
            if not os.path.exists(credentials_path): raise FileNotFoundError("ERRO CRÍTICO: 'credentials.json' não encontrado.")
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, SCOPES); creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token: token.write(creds.to_json())
    try: service = build('drive', 'v3', credentials=creds); print("Serviço do Google Drive conectado."); return service
    except Exception as e: print(f"Erro ao construir o serviço do Drive: {e}"); return None
def find_folder_id(service, folder_name, parent_id='root'):
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); items = results.get('files', []); return items[0]['id'] if items else None
def find_folder_id_by_path(service, path_components):
    current_parent_id = 'root'
    for folder_name in path_components:
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{current_parent_id}' in parents"; results = service.files().list(q=query, fields="files(id)").execute(); items = results.get('files', [])
        if not items: print(f"Pasta '{folder_name}' não encontrada."); return None
        current_parent_id = items[0]['id']
    print("Caminho do dataset encontrado no Drive!"); return current_parent_id
def get_files_from_drive_folder(service, folder_id):
    query = f"'{folder_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); return {file['name']: file['id'] for file in results.get('files', [])}
def download_file_locally(service, file_id, local_filename):
    max_retries = 2 ; attempt = 0
    if os.path.exists(local_filename):
        try: os.remove(local_filename)
        except OSError as e: print(f"  [AVISO Download] Não removeu antigo {local_filename}: {e}")
    while attempt <= max_retries:
        try:
            request = service.files().get_media(fileId=file_id)
            print(f"    Download de {os.path.basename(local_filename)} (tentativa {attempt+1}/{max_retries+1})...", end=' ')
            start_dl = time.time()
            with io.FileIO(local_filename, 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, request); done = False
                while not done: status, done = downloader.next_chunk()
            print(f"OK ({time.time() - start_dl:.2f}s).")
            return True
        except Exception as e:
            attempt += 1; print(f"\n      [AVISO Download] Tentativa {attempt}/{max_retries+1} falhou. Erro: [{type(e).__name__}] {e}")
            if os.path.exists(local_filename):
                try: os.remove(local_filename); print("      Arquivo local corrompido removido.")
                except OSError as remove_error: print(f"      AVISO: Não removeu {local_filename}: {remove_error}")
            if attempt > max_retries: print(f"  [ERRO Download] Download falhou."); return False
            print("      Aguardando 3s..."); time.sleep(3)
    return False
def parse_summary_file(file_path):
    seizure_info_final = {}
    try:
        with open(file_path, 'r', errors='ignore') as f: content = f.read()
    except Exception as e: print(f"  [ERRO] Leitura summary: {file_path} - {e}"); return seizure_info_final
    file_blocks = re.split(r'File Name:\s*', content)
    start_pattern = re.compile(r"Seizure\s*\d*\s*Start Time:\s*(\d+)\s*seconds"); end_pattern = re.compile(r"Seizure\s*\d*\s*End Time:\s*(\d+)\s*seconds")
    for block in file_blocks:
        if not block.strip(): continue
        lines = block.strip().split('\n');
        if not lines: continue
        file_name = lines[0].strip(); current_seizures = []
        for i, line in enumerate(lines):
            start_match = start_pattern.search(line)
            if start_match:
                 start_time = int(start_match.group(1)); end_time = None
                 for k in range(i, min(i + 5, len(lines))):
                      end_match_search = end_pattern.search(lines[k])
                      if end_match_search:
                           seizure_num_start = re.search(r"Seizure\s*(\d*)", line); num_s = seizure_num_start.group(1).strip() if seizure_num_start else ''
                           seizure_num_end = re.search(r"Seizure\s*(\d*)", lines[k]); num_e = seizure_num_end.group(1).strip() if seizure_num_end else ''
                           if num_s == num_e or num_s == '' or num_e == '':
                                end_time = int(end_match_search.group(1))
                                if start_time < end_time: current_seizures.append({'start': start_time, 'end': end_time})
                                else: print(f"  AVISO parse_summary: Ignorando {file_name} crise inválida (start >= end): {start_time} >= {end_time}")
                                break
        if current_seizures:
            seizure_info_final[file_name] = [{'interval': (s['start'], s['end']), 'id': f"{file_name}_s{idx+1}"} for idx, s in enumerate(current_seizures)]
    return seizure_info_final
def extract_single_feature_vector(eeg_window, fs=FS_ASSUMED):
    nperseg = len(eeg_window) if len(eeg_window) > 0 else 1
    try: freqs, psd = welch(eeg_window, fs=fs, nperseg=nperseg)
    except ValueError: psd = np.zeros(nperseg // 2 + 1); freqs = np.linspace(0, fs/2, len(psd))
    total_power = np.sum(psd)
    def get_band_power(f_low, f_high): return np.sum(psd[np.logical_and(freqs >= f_low, freqs <= f_high)])
    delta, theta, alpha, beta, gamma = get_band_power(0.5, 4), get_band_power(4, 8), get_band_power(8, 13), get_band_power(13, 30), get_band_power(30, 80)
    band_powers = [p / total_power if total_power > 0 else 0 for p in [delta, theta, alpha, beta, gamma]]
    ratios = [beta / alpha if alpha > 0 else 0, (delta + theta) / (alpha + beta) if (alpha + beta) > 0 else 0]
    try: pe = antropy.perm_entropy(eeg_window, normalize=True)
    except ValueError: pe = 0
    try: se = antropy.spectral_entropy(eeg_window, sf=fs, method='welch', nperseg=nperseg, normalize=True)
    except (ValueError, OSError): se = 0
    try: sae = antropy.sample_entropy(eeg_window)
    except ValueError: sae = 0
    entropies = [pe, se, sae]
    stats = [np.mean(np.abs(eeg_window)), np.std(eeg_window), skew(eeg_window), kurtosis(eeg_window)]
    try: pfd = antropy.petrosian_fd(eeg_window)
    except (ValueError, ZeroDivisionError): pfd = 0
    fractal_dim = pfd; zero_crossings_rate = antropy.num_zerocross(eeg_window) / len(eeg_window) if len(eeg_window)>0 else 0
    features = [total_power] + band_powers + ratios + entropies + stats + [fractal_dim, zero_crossings_rate]
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
def extract_averaged_features(window_data, fs=FS_ASSUMED):
     n_channels = window_data.shape[0]
     if n_channels == 0: return np.zeros(16)
     channel_features = [extract_single_feature_vector(window_data[i, :], fs) for i in range(n_channels)]
     return np.mean(channel_features, axis=0)
def process_window_wrapper(args):
    window_data, fs = args
    return extract_averaged_features(window_data, fs)

class HDC:
    def __init__(self, dimensions, num_features, num_levels, num_classes=2, seed=None):
        if seed is not None: np.random.seed(seed)
        self.D, self.num_features, self.num_levels, self.num_classes = dimensions, num_features, num_levels, num_classes
        if num_features <= 0: raise ValueError("num_features deve ser > 0")
        self.level_vectors = np.random.choice([-1, 1], size=(num_levels, self.D)); self.feature_vectors = np.random.choice([-1, 1], size=(num_features, self.D))
        self.class_prototypes = np.zeros((self.num_classes, self.D))
        self._sub_prototypes = [] 
        self._sub_proto_labels = [] 
    def _quantize(self, data, num_levels):
        min_val, max_val = np.min(data), np.max(data)
        if max_val - min_val < 1e-9: return np.zeros_like(data, dtype=int)
        quantized = np.round((data - min_val) / (max_val - min_val) * (num_levels - 1))
        return np.clip(quantized, 0, num_levels - 1).astype(int)
    def encode(self, x_data):
        num_samples, num_features = x_data.shape
        if num_features != self.num_features: raise ValueError(f"Dimensão errada: esperado {self.num_features}, recebido {num_features}")
        x_quantized = np.array([self._quantize(x_data[:, i], self.num_levels) for i in range(self.num_features)]).T
        encoded_data_sum = np.sum(self.feature_vectors[None, :, :] * self.level_vectors[x_quantized, :], axis=1)
        return encoded_data_sum 
    def predict(self, x_encoded):
        norm_prototypes = np.sign(self.class_prototypes)
        for k in range(self.num_classes):
             if np.all(norm_prototypes[k] == 0): norm_prototypes[k] = np.random.choice([-1, 1], size=self.D)
        x_encoded_bin = np.sign(x_encoded); x_encoded_bin[np.all(x_encoded_bin == 0, axis=1)] = 0
        return np.argmax(cosine_similarity(x_encoded_bin, norm_prototypes), axis=1)

    def train_multicentroid_multipass(self, x_encoded, y_train, threshold, keep_top_k, epochs, lr, subtract_wrong=True):
        print(f"Iniciando MCri (Th={threshold}, K={keep_top_k}, Ep={epochs}, LR={lr}, Sub={subtract_wrong})...")
        mcri_start_time = time.time()
        self._sub_prototypes = [] 
        self._sub_proto_labels = []
        initial_sub_prototypes_accum = []
        initial_sub_proto_labels = []
        sample_counts = []

        print("  Fase 1a (MC): Criando subs...", end=' ')
        mc_start = time.time()
        for i in range(len(y_train)):
            sample_hv = x_encoded[i]; correct_label = y_train[i]; wrong_label = 1 - correct_label
            sim_correct = -np.inf; best_correct_idx = -1; sim_wrong = -np.inf
            if initial_sub_prototypes_accum:
                sample_hv_bin = np.sign(sample_hv); sample_hv_bin[sample_hv_bin==0] = 1
                current_sub_protos_bin = np.sign(np.array(initial_sub_prototypes_accum))
                for row_idx in range(current_sub_protos_bin.shape[0]):
                    if np.all(current_sub_protos_bin[row_idx] == 0): current_sub_protos_bin[row_idx] = np.random.choice([-1, 1], size=self.D)
                similarities = cosine_similarity(sample_hv_bin.reshape(1, -1), current_sub_protos_bin)[0]
                correct_indices = [idx for idx, lbl in enumerate(initial_sub_proto_labels) if lbl == correct_label]
                if correct_indices:
                    sim_correct_local_idx = np.argmax(similarities[correct_indices]); sim_correct = similarities[correct_indices][sim_correct_local_idx]; best_correct_idx = correct_indices[sim_correct_local_idx]
                wrong_indices = [idx for idx, lbl in enumerate(initial_sub_proto_labels) if lbl == wrong_label]
                if wrong_indices: sim_wrong = np.max(similarities[wrong_indices])
            create_new = (sim_wrong > sim_correct) or (sim_correct == -np.inf)
            if create_new:
                initial_sub_prototypes_accum.append(sample_hv.copy()); initial_sub_proto_labels.append(correct_label); sample_counts.append(1)
            else:
                 initial_sub_prototypes_accum[best_correct_idx] += sample_hv; sample_counts[best_correct_idx] += 1
        print(f"{len(initial_sub_prototypes_accum)} subs ({np.bincount(initial_sub_proto_labels, minlength=2)}). OK ({time.time()-mc_start:.2f}s)")
        if not initial_sub_prototypes_accum: print("AVISO: Nenhum sub criado."); self.class_prototypes = np.zeros((self.num_classes, self.D)); return

        print(f"  Fase 1b (MCr): Reduzindo para ~{keep_top_k}/classe...", end=' ')
        mcr_start = time.time()
        self._sub_prototypes = [] 
        self._sub_proto_labels = []
        for label in range(self.num_classes):
            indices_of_class = [i for i, lbl in enumerate(initial_sub_proto_labels) if lbl == label]
            if not indices_of_class: continue
            counts_of_class = [sample_counts[i] for i in indices_of_class]
            num_to_keep = min(keep_top_k, len(indices_of_class))
            top_k_local_indices = np.argsort(counts_of_class)[-num_to_keep:]
            top_k_global_indices = [indices_of_class[i] for i in top_k_local_indices]
            self._sub_prototypes.extend([initial_sub_prototypes_accum[i] for i in top_k_global_indices])
            self._sub_proto_labels.extend([label] * num_to_keep)
        if not self._sub_prototypes: print("AVISO: Nenhum sub restou."); self.class_prototypes = np.zeros((self.num_classes, self.D)); return
        print(f"{len(self._sub_prototypes)} subs mantidos. OK ({time.time()-mcr_start:.2f}s)")
        self._sub_prototypes = np.array(self._sub_prototypes) 

        print(f"  Fase 2 (MP): Refinando {len(self._sub_prototypes)} subs por {epochs} épocas...", end=' ')
        mp_start = time.time()
        x_encoded_bin = np.sign(x_encoded); x_encoded_bin[np.all(x_encoded_bin == 0, axis=1)] = 1 

        for epoch in range(epochs):
            temp_class_prototypes = np.zeros((self.num_classes, self.D))
            for label in range(self.num_classes):
                indices = [i for i, l in enumerate(self._sub_proto_labels) if l == label]
                if indices: temp_class_prototypes[label] = np.sum(self._sub_prototypes[indices], axis=0) 
            temp_prototypes_norm = np.sign(temp_class_prototypes) 
            for k in range(self.num_classes):
                 if np.all(temp_prototypes_norm[k] == 0): temp_prototypes_norm[k] = np.random.choice([-1, 1], size=self.D)

            y_pred = np.argmax(cosine_similarity(x_encoded_bin, temp_prototypes_norm), axis=1) 
            errors = np.sum(y_pred != y_train)
            if errors == 0: print(f"Conv. {epoch+1} ep. ", end=''); break

            corrections = np.zeros_like(self._sub_prototypes) 
            all_similarities = cosine_similarity(x_encoded, self._sub_prototypes)

            misclassified_idx = np.where(y_pred != y_train)[0]
            if len(misclassified_idx) > 0:
                correct_labels_mc = y_train[misclassified_idx]
                incorrect_labels_mc = y_pred[misclassified_idx]
                samples_mc = x_encoded[misclassified_idx] 
                similarities_mc = all_similarities[misclassified_idx, :]

                closest_correct_global_indices = []
                closest_incorrect_global_indices = []
                for i, mc_idx in enumerate(misclassified_idx):
                    correct_label = correct_labels_mc[i]
                    incorrect_label = incorrect_labels_mc[i]
                    sims_i = similarities_mc[i]

                    correct_indices_local = [idx for idx, label in enumerate(self._sub_proto_labels) if label == correct_label]
                    if correct_indices_local:
                        closest_correct_local_idx = correct_indices_local[np.argmax(sims_i[correct_indices_local])]
                        closest_correct_global_indices.append(closest_correct_local_idx)
                    else: closest_correct_global_indices.append(-1) 

                    incorrect_indices_local = [idx for idx, label in enumerate(self._sub_proto_labels) if label == incorrect_label]
                    if incorrect_indices_local:
                         closest_incorrect_local_idx = incorrect_indices_local[np.argmax(sims_i[incorrect_indices_local])]
                         closest_incorrect_global_indices.append(closest_incorrect_local_idx)
                    else: closest_incorrect_global_indices.append(-1) 

                valid_correct_mask = np.array(closest_correct_global_indices) != -1
                if np.any(valid_correct_mask):
                    np.add.at(corrections, np.array(closest_correct_global_indices)[valid_correct_mask], lr * samples_mc[valid_correct_mask])

                if subtract_wrong:
                    valid_incorrect_mask = np.array(closest_incorrect_global_indices) != -1
                    if np.any(valid_incorrect_mask):
                         np.subtract.at(corrections, np.array(closest_incorrect_global_indices)[valid_incorrect_mask], lr * samples_mc[valid_incorrect_mask])

                self._sub_prototypes += corrections

        print(f"OK ({time.time()-mp_start:.2f}s)")

        print("  Fase 3: Agregando...", end=' ')
        agg_start = time.time()
        final_prototypes = np.zeros((self.num_classes, self.D))
        for label in range(self.num_classes):
            indices = [i for i, l in enumerate(self._sub_proto_labels) if l == label]
            if indices: final_prototypes[label] = np.sum(self._sub_prototypes[indices], axis=0)
        self.class_prototypes = np.sign(final_prototypes) 
        print(f"OK ({time.time()-agg_start:.2f}s). MCri Total: {time.time()-mcri_start_time:.2f}s")


def post_process_smoothing(predictions, smoothing_seconds=5.0, overlap_percentage=0.5):
    window_step_seconds = WINDOW_SECONDS * (1 - overlap_percentage)
    if window_step_seconds <= 0: smoothing_window_size = 1
    else: smoothing_window_size = int(smoothing_seconds / window_step_seconds)
    if smoothing_window_size < 1: smoothing_window_size = 1
    if smoothing_window_size % 2 == 0: smoothing_window_size += 1
    print(f"Aplicando suavização (janela={smoothing_window_size}, {smoothing_seconds}s)...", end=' ')
    start_smooth = time.time()
    if len(predictions) < smoothing_window_size:
        print(f"\n  AVISO: Predições ({len(predictions)}) < Janela ({smoothing_window_size})...")
        smoothing_window_size = max(1, len(predictions));
        if smoothing_window_size > 0 and smoothing_window_size % 2 == 0: smoothing_window_size = max(1, smoothing_window_size -1)
        if smoothing_window_size == 0 : smoothing_window_size = 1
    try: smoothed_predictions = medfilt(predictions, kernel_size=smoothing_window_size)
    except ValueError as e: print(f"\n  AVISO: Erro medfilt: {e}. Retornando originais."); smoothed_predictions = predictions
    print(f"OK ({time.time() - start_smooth:.2f}s)")
    return smoothed_predictions

def main():
    overall_start_time = time.time()
    all_patient_results = []

    try:
        service = get_drive_service()
        root_id = find_folder_id_by_path(service, DRIVE_PATH_COMPONENTS)
        np.random.seed(SEED)

        for patient_idx, patient_name in enumerate(PATIENTS):
            patient_start_time = time.time()
            print(f"\n{'='*20} Iniciando Paciente {patient_idx+1}/{len(PATIENTS)}: {patient_name} {'='*20}")
            patient_folder_id = find_folder_id(service, patient_name, parent_id=root_id);
            if not patient_folder_id: print(f"  Pasta não encontrada. Pulando."); continue
            files_map = get_files_from_drive_folder(service, patient_folder_id)
            all_edf_files_names = sorted([f for f in files_map.keys() if f.endswith('.edf')])
            summary_filename = f"{patient_name}-summary.txt"
            if summary_filename not in files_map: print(f"  AVISO: {summary_filename} não encontrado. Pulando."); continue
            summary_path = f"./temp_{patient_name}_summary.txt";
            if not download_file_locally(service, files_map[summary_filename], summary_path): continue
            seizure_details = parse_summary_file(summary_path); os.remove(summary_path)
            files_with_seizures_set = set(seizure_details.keys())
            if not files_with_seizures_set: print(f"  AVISO: Nenhuma crise válida no summary. Pulando."); continue
            patient_data_by_seizure = defaultdict(lambda: {'features': [], 'labels': []}); seizure_features_list = []
            nonseizure_features_from_seiz_files = []; nonseizure_features_from_other_files = []; file_processing_times = []
            files_to_process_first = [f for f in all_edf_files_names if f in files_with_seizures_set]
            print(f"  Fase 1a: Processando {len(files_to_process_first)} arquivos COM crises...")
            for i_file, edf_name in enumerate(files_to_process_first):
                file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                if not download_file_locally(service, files_map[edf_name], local_path): continue
                try:
                    with pyedflib.EdfReader(local_path) as r:
                        fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                        win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                        current_file_seizures = seizure_details.get(edf_name, [])
                        seizure_intervals_indices = [{'start_idx': int(s['interval'][0] * fs_signal), 'end_idx': int(s['interval'][1] * fs_signal), 'id': s['id']} for s in current_file_seizures]
                        window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file)
                        tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                        print(f"    {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                        feature_extraction_start = time.time(); all_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                        for idx, j in enumerate(window_indices):
                            feats = all_feats[idx]; window_start_idx = j; window_end_idx = j + win_samples_file
                            belongs_to_seizure_id = None; is_seiz = False
                            for seiz_indices in seizure_intervals_indices:
                                s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']
                                if max(window_start_idx, s_idx) < min(window_end_idx, e_idx): is_seiz = True; belongs_to_seizure_id = seiz_indices['id']; break
                            if is_seiz:
                                patient_data_by_seizure[belongs_to_seizure_id]['features'].append(feats); patient_data_by_seizure[belongs_to_seizure_id]['labels'].append(1); seizure_features_list.append(feats)
                            else:
                                too_close = False
                                for seiz_indices in seizure_intervals_indices:
                                    s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']; exclusion_start_idx = s_idx - int(60 * fs_signal); exclusion_end_idx = e_idx + int(15 * 60 * fs_signal)
                                    if max(window_start_idx, exclusion_start_idx) < min(window_end_idx, exclusion_end_idx): too_close = True; break
                                if not too_close: nonseizure_features_from_seiz_files.append(feats)
                except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                finally:
                    if os.path.exists(local_path): os.remove(local_path)
                gc.collect(); file_end_time = time.time(); file_processing_times.append(file_end_time - file_start_time); avg_time_per_file = np.mean(file_processing_times) if file_processing_times else 0
                remaining_files = len(files_to_process_first) - (i_file + 1); eta_sec = remaining_files * avg_time_per_file
                print(f"    {i_file+1}/{len(files_to_process_first)}: {edf_name} processado em {time.time() - file_start_time:.2f}s. ETA: {eta_sec/60:.1f} min")
            num_seiz_total = len(seizure_features_list)
            if num_seiz_total == 0: print(f"  AVISO: Nenhuma JANELA de crise detectada. Pulando."); continue
            num_nonseiz_found_initial = len(nonseizure_features_from_seiz_files); num_nonseiz_needed_total = num_seiz_total * 10
            print(f"  Fase 1a concluída: {num_seiz_total} janelas crise, {num_nonseiz_found_initial} não-crise."); print(f"                     Necessárias {num_nonseiz_needed_total}.")
            nonseizure_features_from_other_files = [] 
            if num_nonseiz_found_initial < num_nonseiz_needed_total:
                deficit = num_nonseiz_needed_total - num_nonseiz_found_initial; print(f"  Fase 1b: Buscando mais {deficit} não-crises...")
                files_to_process_later = [f for f in all_edf_files_names if f not in files_with_seizures_set]
                if not files_to_process_later: print("  AVISO: Não há outros arquivos.")
                else:
                    print(f"           Processando até {len(files_to_process_later)} arquivos (paralelo)...")
                    nonseiz_collected_later = 0; phase1b_file_times = []
                    for i_file, edf_name in enumerate(files_to_process_later):
                        if nonseiz_collected_later >= deficit: break
                        file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                        if not download_file_locally(service, files_map[edf_name], local_path): continue
                        try:
                            with pyedflib.EdfReader(local_path) as r:
                                fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                                win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                                window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file); tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                                print(f"      {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                                feature_extraction_start = time.time(); other_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                                needed_now = deficit - nonseiz_collected_later; added_now = min(needed_now, len(other_feats)); nonseizure_features_from_other_files.extend(other_feats[:added_now]); nonseiz_collected_later += added_now
                        except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                        finally:
                            if os.path.exists(local_path): os.remove(local_path)
                        gc.collect(); file_end_time = time.time(); phase1b_file_times.append(file_end_time - file_start_time); avg_time_b = np.mean(phase1b_file_times) if phase1b_file_times else 0; remaining_files_b = len(files_to_process_later) - (i_file + 1)
                        files_really_needed = np.ceil(deficit / (nonseiz_collected_later / (i_file + 1))) if (i_file+1)>0 and nonseiz_collected_later>0 else remaining_files_b
                        eta_sec_b = min(remaining_files_b, max(0, files_really_needed - (i_file+1))) * avg_time_b
                        print(f"    {i_file+1}/{len(files_to_process_later)}: {edf_name} em {time.time() - file_start_time:.2f}s ({nonseiz_collected_later}/{deficit}). ETA: {eta_sec_b/60:.1f} min")
                    print(f"  Fase 1b concluída: {nonseiz_collected_later} janelas adicionais.")
            else: print("  Não foi necessário Fase 1b.")
            patient_potential_nonseizure_features = nonseizure_features_from_seiz_files + nonseizure_features_from_other_files
            num_nonseiz_available = len(patient_potential_nonseizure_features)
            if num_nonseiz_available == 0: print(f"  AVISO CRÍTICO: Nenhuma janela não-crise válida. Pulando."); continue
            if num_nonseiz_available >= num_nonseiz_needed_total:
                indices = np.random.choice(num_nonseiz_available, num_nonseiz_needed_total, replace=False)
                selected_nonseizure_features = [patient_potential_nonseizure_features[i] for i in indices]
            else: selected_nonseizure_features = patient_potential_nonseizure_features; print(f"  AVISO FINAL: Apenas {len(selected_nonseizure_features)}/{num_nonseiz_needed_total} não-crises.")
            num_nonseiz_final = len(selected_nonseizure_features)
            print(f"  Dataset Paciente: {num_seiz_total} crises, {num_nonseiz_final} não-crises (Rácio ~1:{num_nonseiz_final/num_seiz_total:.1f}).")

            seizure_ids = list(patient_data_by_seizure.keys())
            if not seizure_ids or len(seizure_ids) < 2: print(f"  AVISO: Crises insuficientes ({len(seizure_ids)}) para LOSO. Pulando."); continue
            patient_fold_metrics = []
            print(f"  Fase 2: Iniciando LOSO-CV com {len(seizure_ids)} crises...")
            for k, seizure_id_out in enumerate(seizure_ids):
                fold_start_time = time.time()
                print(f"\n    --- FOLD {k+1}/{len(seizure_ids)} ({seizure_id_out} fora) ---")
                X_train_list, y_train_list = [], []; X_test_list, y_test_list = [], []
                test_seizure_data = patient_data_by_seizure[seizure_id_out]; num_seiz_test = len(test_seizure_data['features'])
                if num_seiz_test == 0: print(f"      AVISO: Crise {seizure_id_out} sem janelas. Pulando."); continue
                X_test_list.extend(test_seizure_data['features']); y_test_list.extend([1] * num_seiz_test)
                num_nonseiz_needed_test = num_seiz_test * 10
                if num_nonseiz_final >= num_nonseiz_needed_test:
                     test_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_test, replace=False)
                     X_test_list.extend([selected_nonseizure_features[i] for i in test_nonseiz_indices]); y_test_list.extend([0] * num_nonseiz_needed_test)
                else: X_test_list.extend(selected_nonseizure_features); y_test_list.extend([0] * num_nonseiz_final)
                total_seiz_train = 0
                for seizure_id_train, data in patient_data_by_seizure.items():
                    if seizure_id_train != seizure_id_out:
                        num_windows_seiz_train = len(data['features'])
                        if num_windows_seiz_train > 0: X_train_list.extend(data['features']); y_train_list.extend([1] * num_windows_seiz_train); total_seiz_train += num_windows_seiz_train
                if total_seiz_train == 0: print(f"      AVISO: Sem crises para treino. Pulando."); continue
                num_nonseiz_needed_train = total_seiz_train * 10
                if num_nonseiz_final >= num_nonseiz_needed_train:
                    train_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_train, replace=False)
                    X_train_list.extend([selected_nonseizure_features[i] for i in train_nonseiz_indices]); y_train_list.extend([0] * num_nonseiz_needed_train)
                else: X_train_list.extend(selected_nonseizure_features); y_train_list.extend([0] * num_nonseiz_final)
                X_train, y_train = np.array(X_train_list), np.array(y_train_list); X_test, y_test = np.array(X_test_list), np.array(y_test_list)
                if X_train.ndim != 2 or X_train.shape[1] == 0: print(f"      AVISO: X_train inválido. Shape={X_train.shape}. Pulando."); continue
                print(f"      Treino: {len(X_train)} ({np.bincount(y_train, minlength=2)}), Teste: {len(X_test)} ({np.bincount(y_test, minlength=2)})")
                if len(np.unique(y_train)) < 2 or len(X_test) == 0: print(f"      AVISO: Dados insuficientes. Pulando."); continue

                scaler = StandardScaler(); X_train_scaled = scaler.fit_transform(X_train); X_test_scaled = scaler.transform(X_test)
                num_features_fold = X_train_scaled.shape[1]
                model = HDC(DIMENSIONS, num_features_fold, NUM_LEVELS, seed=SEED)
                encode_start_time = time.time();
                X_train_hd = model.encode(X_train_scaled);
                X_test_hd = model.encode(X_test_scaled);
                print(f"      Encoding: {time.time() - encode_start_time:.2f}s")

                model.train_multicentroid_multipass(
                    X_train_hd, y_train, threshold=MC_THRESHOLD, keep_top_k=MCr_KEEP_TOP_K,
                    epochs=EPOCHS, lr=LEARNING_RATE, subtract_wrong=SUBTRACT_WRONG
                )

                predict_start_time = time.time(); y_pred_raw = model.predict(X_test_hd); print(f"      Predição: {time.time() - predict_start_time:.2f}s") 
                y_pred_processed = post_process_smoothing(y_pred_raw, smoothing_seconds=SMOOTHING_SECONDS, overlap_percentage=OVERLAP_PERCENTAGE)
                acc = accuracy_score(y_test, y_pred_processed); f1 = f1_score(y_test, y_pred_processed, zero_division=0); precision = precision_score(y_test, y_pred_processed, zero_division=0); recall = recall_score(y_test, y_pred_processed, zero_division=0)
                patient_fold_metrics.append({'acc': acc, 'f1': f1, 'precision': precision, 'recall': recall})
                print(f"      Resultados: Acc={acc:.3f}, F1={f1:.3f}, P={precision:.3f}, R={recall:.3f}")
                cm = confusion_matrix(y_test, y_pred_processed, labels=[0, 1]); print(f"      Matriz (TN, FP / FN, TP):\n{cm}")
                gc.collect(); print(f"    Fold {k+1} concluído em {time.time() - fold_start_time:.2f}s")

            if patient_fold_metrics:
                avg_acc_pat = np.mean([m['acc'] for m in patient_fold_metrics]); avg_f1_pat = np.mean([m['f1'] for m in patient_fold_metrics]); avg_precision_pat = np.mean([m['precision'] for m in patient_fold_metrics]); avg_recall_pat = np.mean([m['recall'] for m in patient_fold_metrics])
                print(f"\n  Resultado Médio {patient_name}: Acc={avg_acc_pat:.3f}, F1={avg_f1_pat:.3f}, P={avg_precision_pat:.3f}, R={avg_recall_pat:.3f}")
                all_patient_results.append({'acc': avg_acc_pat, 'f1': avg_f1_pat, 'precision': avg_precision_pat, 'recall': avg_recall_pat})
            else: print(f"  Nenhum fold LOSO concluído para {patient_name}.")
            patient_end_time = time.time(); print(f"  Tempo Paciente {patient_name}: {(patient_end_time - patient_start_time) / 60:.2f} min"); gc.collect()

        if all_patient_results:
            print("\n\n" + "="*40 + "\n" + "RESULTADO FINAL (MÉDIA ENTRE PACIENTES)".center(40) + "\n" + "="*40)
            avg_acc_all = np.mean([m['acc'] for m in all_patient_results]); std_acc_all = np.std([m['acc'] for m in all_patient_results])
            avg_f1_all = np.mean([m['f1'] for m in all_patient_results]); std_f1_all = np.std([m['f1'] for m in all_patient_results])
            avg_precision_all = np.mean([m['precision'] for m in all_patient_results]); std_precision_all = np.std([m['precision'] for m in all_patient_results])
            avg_recall_all = np.mean([m['recall'] for m in all_patient_results]); std_recall_all = np.std([m['recall'] for m in all_patient_results])
            print(f"Acurácia Média Geral:      {avg_acc_all:.3f} +/- {std_acc_all:.3f}")
            print(f"F1 Score Médio Geral:      {avg_f1_all:.3f} +/- {std_f1_all:.3f}")
            print(f"Precisão Média Geral:      {avg_precision_all:.3f} +/- {std_precision_all:.3f}")
            print(f"Recall (Sens.) Médio Geral: {avg_recall_all:.3f} +/- {std_recall_all:.3f}")
        else: print("\nNenhum resultado de paciente para média geral.")

    except Exception as e: print(f"\nERRO GERAL: {e}")
    finally:
        overall_end_time = time.time(); print(f"\nTempo total: {(overall_end_time - overall_start_time) / 60:.2f} min", flush=True)

if __name__ == '__main__':
    main()

Serviço do Google Drive conectado.
Caminho do dataset encontrado no Drive!

==================== Iniciando Paciente 1/1: chb01 ====================
    Download de temp_chb01_summary.txt (tentativa 1/3)... OK (1.25s).
  Fase 1a: Processando 7 arquivos COM crises...
    Download de temp_chb01_03.edf (tentativa 1/3)... OK (21.08s).
    chb01_03.edf: Extraindo features 1439 janelas (-1 cores)... OK (97.38s)
    1/7: chb01_03.edf processado em 121.18s. ETA: 12.1 min
    Download de temp_chb01_04.edf (tentativa 1/3)... OK (41.82s).
    chb01_04.edf: Extraindo features 1439 janelas (-1 cores)... OK (97.48s)
    2/7: chb01_04.edf processado em 141.93s. ETA: 11.0 min
    Download de temp_chb01_15.edf (tentativa 1/3)... OK (44.32s).
    chb01_15.edf: Extraindo features 1439 janelas (-1 cores)... OK (100.26s)
    3/7: chb01_15.edf processado em 147.10s. ETA: 9.1 min
    Download de temp_chb01_16.edf (tentativa 1/3)... OK (38.70s).
    chb01_16.edf: Extraindo features 1439 janelas (-1 cores)... O

##### MODELOS CLASSICOS

In [ ]:
import numpy as np
import os
import re
import time
import io
import gc
import warnings
from scipy.signal import welch, medfilt
from scipy.stats import skew, kurtosis
import antropy
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pyedflib
from collections import defaultdict
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=RuntimeWarning)
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

PATIENTS = [f"chb{str(i).zfill(2)}" for i in range(1, 2)] 
DRIVE_PATH_COMPONENTS = ['TCC EPILEPSIA DATA', 'chb-mit-scalp-eeg-database-1.0.0']
MAX_CHANNELS, SEED = 23, 42
WINDOW_SECONDS, OVERLAP_PERCENTAGE = 5.0, 0.5
SMOOTHING_SECONDS = 5.0
FS_ASSUMED = 256
N_JOBS = -1 

def get_drive_service():
    creds = None; creds_folder = 'credentials'; token_path = os.path.join(creds_folder, 'token.json'); credentials_path = os.path.join(creds_folder, 'credentials.json'); os.makedirs(creds_folder, exist_ok=True)
    if os.path.exists(token_path): creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token: creds.refresh(Request())
        else:
            if not os.path.exists(credentials_path): raise FileNotFoundError("ERRO CRÍTICO: 'credentials.json' não encontrado.")
            flow = InstalledAppFlow.from_client_secrets_file(credentials_path, SCOPES); creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token: token.write(creds.to_json())
    try: service = build('drive', 'v3', credentials=creds); print("Serviço do Google Drive conectado."); return service
    except Exception as e: print(f"Erro ao construir o serviço do Drive: {e}"); return None
def find_folder_id(service, folder_name, parent_id='root'):
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); items = results.get('files', []); return items[0]['id'] if items else None
def find_folder_id_by_path(service, path_components):
    current_parent_id = 'root'
    for folder_name in path_components:
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{current_parent_id}' in parents"; results = service.files().list(q=query, fields="files(id)").execute(); items = results.get('files', [])
        if not items: print(f"Pasta '{folder_name}' não encontrada."); return None
        current_parent_id = items[0]['id']
    print("Caminho do dataset encontrado no Drive!"); return current_parent_id
def get_files_from_drive_folder(service, folder_id):
    query = f"'{folder_id}' in parents"; results = service.files().list(q=query, fields="files(id, name)").execute(); return {file['name']: file['id'] for file in results.get('files', [])}
def download_file_locally(service, file_id, local_filename):
    max_retries = 2 ; attempt = 0
    if os.path.exists(local_filename):
        try: os.remove(local_filename)
        except OSError as e: print(f"  [AVISO Download] Não removeu antigo {local_filename}: {e}")
    while attempt <= max_retries:
        try:
            request = service.files().get_media(fileId=file_id)
            print(f"    Download de {os.path.basename(local_filename)} (tentativa {attempt+1}/{max_retries+1})...", end=' ')
            start_dl = time.time()
            with io.FileIO(local_filename, 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, request); done = False
                while not done: status, done = downloader.next_chunk()
            print(f"OK ({time.time() - start_dl:.2f}s).")
            return True
        except Exception as e:
            attempt += 1; print(f"\n      [AVISO Download] Tentativa {attempt}/{max_retries+1} falhou. Erro: [{type(e).__name__}] {e}")
            if os.path.exists(local_filename):
                try: os.remove(local_filename); print("      Arquivo local corrompido removido.")
                except OSError as remove_error: print(f"      AVISO: Não removeu {local_filename}: {remove_error}")
            if attempt > max_retries: print(f"  [ERRO Download] Download falhou."); return False
            print("      Aguardando 3s..."); time.sleep(3)
    return False
def parse_summary_file(file_path):
    seizure_info_final = {}
    try:
        with open(file_path, 'r', errors='ignore') as f: content = f.read()
    except Exception as e: print(f"  [ERRO] Leitura summary: {file_path} - {e}"); return seizure_info_final
    file_blocks = re.split(r'File Name:\s*', content)
    start_pattern = re.compile(r"Seizure\s*\d*\s*Start Time:\s*(\d+)\s*seconds"); end_pattern = re.compile(r"Seizure\s*\d*\s*End Time:\s*(\d+)\s*seconds")
    for block in file_blocks:
        if not block.strip(): continue
        lines = block.strip().split('\n');
        if not lines: continue
        file_name = lines[0].strip(); current_seizures = []
        for i, line in enumerate(lines):
            start_match = start_pattern.search(line)
            if start_match:
                 start_time = int(start_match.group(1)); end_time = None
                 for k in range(i, min(i + 5, len(lines))):
                      end_match_search = end_pattern.search(lines[k])
                      if end_match_search:
                           seizure_num_start = re.search(r"Seizure\s*(\d*)", line); num_s = seizure_num_start.group(1).strip() if seizure_num_start else ''
                           seizure_num_end = re.search(r"Seizure\s*(\d*)", lines[k]); num_e = seizure_num_end.group(1).strip() if seizure_num_end else ''
                           if num_s == num_e or num_s == '' or num_e == '':
                                end_time = int(end_match_search.group(1))
                                if start_time < end_time: current_seizures.append({'start': start_time, 'end': end_time})
                                else: print(f"  AVISO parse_summary: Ignorando {file_name} crise inválida (start >= end): {start_time} >= {end_time}")
                                break
        if current_seizures:
            seizure_info_final[file_name] = [{'interval': (s['start'], s['end']), 'id': f"{file_name}_s{idx+1}"} for idx, s in enumerate(current_seizures)]
    return seizure_info_final
def extract_single_feature_vector(eeg_window, fs=FS_ASSUMED):
    nperseg = len(eeg_window) if len(eeg_window) > 0 else 1
    try: freqs, psd = welch(eeg_window, fs=fs, nperseg=nperseg)
    except ValueError: psd = np.zeros(nperseg // 2 + 1); freqs = np.linspace(0, fs/2, len(psd))
    total_power = np.sum(psd)
    def get_band_power(f_low, f_high): return np.sum(psd[np.logical_and(freqs >= f_low, freqs <= f_high)])
    delta, theta, alpha, beta, gamma = get_band_power(0.5, 4), get_band_power(4, 8), get_band_power(8, 13), get_band_power(13, 30), get_band_power(30, 80)
    band_powers = [p / total_power if total_power > 0 else 0 for p in [delta, theta, alpha, beta, gamma]]
    ratios = [beta / alpha if alpha > 0 else 0, (delta + theta) / (alpha + beta) if (alpha + beta) > 0 else 0]
    try: pe = antropy.perm_entropy(eeg_window, normalize=True)
    except ValueError: pe = 0
    try: se = antropy.spectral_entropy(eeg_window, sf=fs, method='welch', nperseg=nperseg, normalize=True)
    except (ValueError, OSError): se = 0
    try: sae = antropy.sample_entropy(eeg_window)
    except ValueError: sae = 0
    entropies = [pe, se, sae]
    stats = [np.mean(np.abs(eeg_window)), np.std(eeg_window), skew(eeg_window), kurtosis(eeg_window)]
    try: pfd = antropy.petrosian_fd(eeg_window)
    except (ValueError, ZeroDivisionError): pfd = 0
    fractal_dim = pfd; zero_crossings_rate = antropy.num_zerocross(eeg_window) / len(eeg_window) if len(eeg_window)>0 else 0
    features = [total_power] + band_powers + ratios + entropies + stats + [fractal_dim, zero_crossings_rate]
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
def extract_averaged_features(window_data, fs=FS_ASSUMED):
     n_channels = window_data.shape[0]
     if n_channels == 0: return np.zeros(16)
     channel_features = [extract_single_feature_vector(window_data[i, :], fs) for i in range(n_channels)]
     return np.mean(channel_features, axis=0)
def process_window_wrapper(args):
    window_data, fs = args
    return extract_averaged_features(window_data, fs)
def post_process_smoothing(predictions, smoothing_seconds=5.0, overlap_percentage=0.5):
    window_step_seconds = WINDOW_SECONDS * (1 - overlap_percentage)
    if window_step_seconds <= 0: smoothing_window_size = 1
    else: smoothing_window_size = int(smoothing_seconds / window_step_seconds)
    if smoothing_window_size < 1: smoothing_window_size = 1
    if smoothing_window_size % 2 == 0: smoothing_window_size += 1
    print(f"Aplicando suavização (janela={smoothing_window_size}, {smoothing_seconds}s)...", end=' ')
    start_smooth = time.time()
    if len(predictions) < smoothing_window_size:
        print(f"\n  AVISO: Predições ({len(predictions)}) < Janela ({smoothing_window_size})...")
        smoothing_window_size = max(1, len(predictions));
        if smoothing_window_size > 0 and smoothing_window_size % 2 == 0: smoothing_window_size = max(1, smoothing_window_size -1)
        if smoothing_window_size == 0 : smoothing_window_size = 1
    try: smoothed_predictions = medfilt(predictions, kernel_size=smoothing_window_size)
    except ValueError as e: print(f"\n  AVISO: Erro medfilt: {e}. Retornando originais."); smoothed_predictions = predictions
    print(f"OK ({time.time() - start_smooth:.2f}s)")
    return smoothed_predictions

def main():
    overall_start_time = time.time()
    all_models_results = defaultdict(lambda: {'acc': [], 'f1': [], 'precision': [], 'recall': []})

    try:
        service = get_drive_service()
        root_id = find_folder_id_by_path(service, DRIVE_PATH_COMPONENTS)
        np.random.seed(SEED)

        models_to_evaluate = {
            'RandomForest': RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=N_JOBS, class_weight='balanced'),
            'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
            'SVM': SVC(random_state=SEED, kernel='rbf', class_weight='balanced', probability=False),
            'LogisticRegression': LogisticRegression(random_state=SEED, max_iter=1000, class_weight='balanced', n_jobs=N_JOBS, solver='liblinear'),
            'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=N_JOBS),
            'DecisionTree': DecisionTreeClassifier(random_state=SEED, class_weight='balanced'),
        }

        for patient_idx, patient_name in enumerate(PATIENTS):
            patient_start_time = time.time()
            print(f"\n{'='*20} Iniciando Paciente {patient_idx+1}/{len(PATIENTS)}: {patient_name} {'='*20}")
            patient_folder_id = find_folder_id(service, patient_name, parent_id=root_id);
            if not patient_folder_id: print(f"  Pasta não encontrada. Pulando."); continue
            files_map = get_files_from_drive_folder(service, patient_folder_id)
            all_edf_files_names = sorted([f for f in files_map.keys() if f.endswith('.edf')])
            summary_filename = f"{patient_name}-summary.txt"
            if summary_filename not in files_map: print(f"  AVISO: {summary_filename} não encontrado. Pulando."); continue
            summary_path = f"./temp_{patient_name}_summary.txt";
            if not download_file_locally(service, files_map[summary_filename], summary_path): continue
            seizure_details = parse_summary_file(summary_path); os.remove(summary_path)
            files_with_seizures_set = set(seizure_details.keys())
            if not files_with_seizures_set: print(f"  AVISO: Nenhuma crise válida no summary. Pulando."); continue
            patient_data_by_seizure = defaultdict(lambda: {'features': [], 'labels': []}); seizure_features_list = []
            nonseizure_features_from_seiz_files = []; nonseizure_features_from_other_files = []; file_processing_times = []
            files_to_process_first = [f for f in all_edf_files_names if f in files_with_seizures_set]
            print(f"  Fase 1a: Processando {len(files_to_process_first)} arquivos COM crises...")
            for i_file, edf_name in enumerate(files_to_process_first):
                file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                if not download_file_locally(service, files_map[edf_name], local_path): continue
                try:
                    with pyedflib.EdfReader(local_path) as r:
                        fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                        win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                        current_file_seizures = seizure_details.get(edf_name, [])
                        seizure_intervals_indices = [{'start_idx': int(s['interval'][0] * fs_signal), 'end_idx': int(s['interval'][1] * fs_signal), 'id': s['id']} for s in current_file_seizures]
                        window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file)
                        tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                        print(f"    {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                        feature_extraction_start = time.time(); all_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                        for idx, j in enumerate(window_indices):
                            feats = all_feats[idx]; window_start_idx = j; window_end_idx = j + win_samples_file
                            belongs_to_seizure_id = None; is_seiz = False
                            for seiz_indices in seizure_intervals_indices:
                                s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']
                                if max(window_start_idx, s_idx) < min(window_end_idx, e_idx): is_seiz = True; belongs_to_seizure_id = seiz_indices['id']; break
                            if is_seiz:
                                patient_data_by_seizure[belongs_to_seizure_id]['features'].append(feats); patient_data_by_seizure[belongs_to_seizure_id]['labels'].append(1); seizure_features_list.append(feats)
                            else:
                                too_close = False
                                for seiz_indices in seizure_intervals_indices:
                                    s_idx, e_idx = seiz_indices['start_idx'], seiz_indices['end_idx']; exclusion_start_idx = s_idx - int(60 * fs_signal); exclusion_end_idx = e_idx + int(15 * 60 * fs_signal)
                                    if max(window_start_idx, exclusion_start_idx) < min(window_end_idx, exclusion_end_idx): too_close = True; break
                                if not too_close: nonseizure_features_from_seiz_files.append(feats)
                except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                finally:
                    if os.path.exists(local_path): os.remove(local_path)
                gc.collect(); file_end_time = time.time(); file_processing_times.append(file_end_time - file_start_time); avg_time_per_file = np.mean(file_processing_times) if file_processing_times else 0
                remaining_files = len(files_to_process_first) - (i_file + 1); eta_sec = remaining_files * avg_time_per_file
                print(f"    {i_file+1}/{len(files_to_process_first)}: {edf_name} processado em {time.time() - file_start_time:.2f}s. ETA: {eta_sec/60:.1f} min")
            num_seiz_total = len(seizure_features_list)
            if num_seiz_total == 0: print(f"  AVISO: Nenhuma JANELA de crise detectada. Pulando."); continue
            num_nonseiz_found_initial = len(nonseizure_features_from_seiz_files); num_nonseiz_needed_total = num_seiz_total * 10
            print(f"  Fase 1a concluída: {num_seiz_total} janelas crise, {num_nonseiz_found_initial} não-crise."); print(f"                     Necessárias {num_nonseiz_needed_total}.")
            nonseizure_features_from_other_files = []
            if num_nonseiz_found_initial < num_nonseiz_needed_total:
                deficit = num_nonseiz_needed_total - num_nonseiz_found_initial; print(f"  Fase 1b: Buscando mais {deficit} não-crises...")
                files_to_process_later = [f for f in all_edf_files_names if f not in files_with_seizures_set]
                if not files_to_process_later: print("  AVISO: Não há outros arquivos.")
                else:
                    print(f"           Processando até {len(files_to_process_later)} arquivos (paralelo)...")
                    nonseiz_collected_later = 0; phase1b_file_times = []
                    for i_file, edf_name in enumerate(files_to_process_later):
                        if nonseiz_collected_later >= deficit: break
                        file_start_time = time.time(); local_path = f"./temp_{edf_name}"
                        if not download_file_locally(service, files_map[edf_name], local_path): continue
                        try:
                            with pyedflib.EdfReader(local_path) as r:
                                fs_signal = r.getSampleFrequency(0); n_ch = min(r.signals_in_file, MAX_CHANNELS); signals = np.array([r.readSignal(c) for c in range(n_ch)], dtype=np.float32)
                                win_samples_file = int(fs_signal * WINDOW_SECONDS); step_file = int(win_samples_file * (1.0 - OVERLAP_PERCENTAGE)); step_file = max(1, step_file)
                                window_indices = range(0, signals.shape[1] - win_samples_file + 1, step_file); tasks = [(signals[:, j : j + win_samples_file], fs_signal) for j in window_indices]
                                print(f"      {edf_name}: Extraindo features {len(tasks)} janelas ({N_JOBS} cores)...", end=' ')
                                feature_extraction_start = time.time(); other_feats = Parallel(n_jobs=N_JOBS)(delayed(process_window_wrapper)(task) for task in tasks); print(f"OK ({time.time() - feature_extraction_start:.2f}s)")
                                needed_now = deficit - nonseiz_collected_later; added_now = min(needed_now, len(other_feats)); nonseizure_features_from_other_files.extend(other_feats[:added_now]); nonseiz_collected_later += added_now
                        except Exception as e: print(f"  [ERRO] Falha ao processar {edf_name}: {e}")
                        finally:
                            if os.path.exists(local_path): os.remove(local_path)
                        gc.collect(); file_end_time = time.time(); phase1b_file_times.append(file_end_time - file_start_time); avg_time_b = np.mean(phase1b_file_times) if phase1b_file_times else 0; remaining_files_b = len(files_to_process_later) - (i_file + 1)
                        files_really_needed = np.ceil(deficit / (nonseiz_collected_later / (i_file + 1))) if (i_file+1)>0 and nonseiz_collected_later>0 else remaining_files_b
                        eta_sec_b = min(remaining_files_b, max(0, files_really_needed - (i_file+1))) * avg_time_b
                        print(f"    {i_file+1}/{len(files_to_process_later)}: {edf_name} em {time.time() - file_start_time:.2f}s ({nonseiz_collected_later}/{deficit}). ETA: {eta_sec_b/60:.1f} min")
                    print(f"  Fase 1b concluída: {nonseiz_collected_later} janelas adicionais.")
            else: print("  Não foi necessário Fase 1b.")
            patient_potential_nonseizure_features = nonseizure_features_from_seiz_files + nonseizure_features_from_other_files
            num_nonseiz_available = len(patient_potential_nonseizure_features)
            if num_nonseiz_available == 0: print(f"  AVISO CRÍTICO: Nenhuma janela não-crise válida. Pulando."); continue
            if num_nonseiz_available >= num_nonseiz_needed_total:
                indices = np.random.choice(num_nonseiz_available, num_nonseiz_needed_total, replace=False)
                selected_nonseizure_features = [patient_potential_nonseizure_features[i] for i in indices]
            else: selected_nonseizure_features = patient_potential_nonseizure_features; print(f"  AVISO FINAL: Apenas {len(selected_nonseizure_features)}/{num_nonseiz_needed_total} não-crises.")
            num_nonseiz_final = len(selected_nonseizure_features)
            print(f"  Dataset Paciente: {num_seiz_total} crises, {num_nonseiz_final} não-crises (Rácio ~1:{num_nonseiz_final/num_seiz_total:.1f}).")

            seizure_ids = list(patient_data_by_seizure.keys())
            if not seizure_ids or len(seizure_ids) < 2: print(f"  AVISO: Crises insuficientes ({len(seizure_ids)}) para LOSO. Pulando."); continue
            patient_model_fold_metrics = defaultdict(list)
            print(f"  Fase 2: Iniciando LOSO-CV com {len(seizure_ids)} crises...")
            for k, seizure_id_out in enumerate(seizure_ids):
                fold_start_time = time.time()
                print(f"\n    --- FOLD {k+1}/{len(seizure_ids)} ({seizure_id_out} fora) ---")
                X_train_list, y_train_list = [], []; X_test_list, y_test_list = [], []
                test_seizure_data = patient_data_by_seizure[seizure_id_out]; num_seiz_test = len(test_seizure_data['features'])
                if num_seiz_test == 0: print(f"      AVISO: Crise {seizure_id_out} sem janelas. Pulando."); continue
                X_test_list.extend(test_seizure_data['features']); y_test_list.extend([1] * num_seiz_test)
                num_nonseiz_needed_test = num_seiz_test * 10
                if num_nonseiz_final >= num_nonseiz_needed_test:
                     test_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_test, replace=False)
                     X_test_list.extend([selected_nonseizure_features[i] for i in test_nonseiz_indices]); y_test_list.extend([0] * num_nonseiz_needed_test)
                else: X_test_list.extend(selected_nonseizure_features); y_test_list.extend([0] * num_nonseiz_final)
                total_seiz_train = 0
                for seizure_id_train, data in patient_data_by_seizure.items():
                    if seizure_id_train != seizure_id_out:
                        num_windows_seiz_train = len(data['features'])
                        if num_windows_seiz_train > 0: X_train_list.extend(data['features']); y_train_list.extend([1] * num_windows_seiz_train); total_seiz_train += num_windows_seiz_train
                if total_seiz_train == 0: print(f"      AVISO: Sem crises para treino. Pulando."); continue
                num_nonseiz_needed_train = total_seiz_train * 10
                if num_nonseiz_final >= num_nonseiz_needed_train:
                    train_nonseiz_indices = np.random.choice(num_nonseiz_final, num_nonseiz_needed_train, replace=False)
                    X_train_list.extend([selected_nonseizure_features[i] for i in train_nonseiz_indices]); y_train_list.extend([0] * num_nonseiz_needed_train)
                else: X_train_list.extend(selected_nonseizure_features); y_train_list.extend([0] * num_nonseiz_final)
                X_train, y_train = np.array(X_train_list), np.array(y_train_list); X_test, y_test = np.array(X_test_list), np.array(y_test_list)
                if X_train.ndim != 2 or X_train.shape[1] == 0: print(f"      AVISO: X_train inválido. Shape={X_train.shape}. Pulando."); continue
                print(f"      Treino: {len(X_train)} ({np.bincount(y_train, minlength=2)}), Teste: {len(X_test)} ({np.bincount(y_test, minlength=2)})")
                if len(np.unique(y_train)) < 2 or len(X_test) == 0: print(f"      AVISO: Dados insuficientes. Pulando."); continue

                scaler = StandardScaler(); X_train_scaled = scaler.fit_transform(X_train); X_test_scaled = scaler.transform(X_test)

                for model_name, model_template in models_to_evaluate.items():
                    print(f"      Treinando {model_name}...", end=' ')
                    train_start_time = time.time()
                    model = model_template
                    if 'estimator' in model_name: 
                         model = type(model_template)() 
                         model.set_params(**model_template.get_params()) 

                    model.fit(X_train_scaled, y_train)
                    print(f"OK ({time.time() - train_start_time:.2f}s). ", end='')

                    predict_start_time = time.time(); y_pred_raw = model.predict(X_test_scaled); print(f"Predição: {time.time() - predict_start_time:.2f}s. ", end='')
                    y_pred_processed = post_process_smoothing(y_pred_raw, smoothing_seconds=SMOOTHING_SECONDS, overlap_percentage=OVERLAP_PERCENTAGE)
                    acc = accuracy_score(y_test, y_pred_processed); f1 = f1_score(y_test, y_pred_processed, zero_division=0); precision = precision_score(y_test, y_pred_processed, zero_division=0); recall = recall_score(y_test, y_pred_processed, zero_division=0)
                    patient_model_fold_metrics[model_name].append({'acc': acc, 'f1': f1, 'precision': precision, 'recall': recall})
                    print(f"      Resultados {model_name}: Acc={acc:.3f}, F1={f1:.3f}, P={precision:.3f}, R={recall:.3f}")
                    cm = confusion_matrix(y_test, y_pred_processed, labels=[0, 1]); 
                gc.collect(); print(f"    Fold {k+1} concluído em {time.time() - fold_start_time:.2f}s")

            print(f"\n  Resultados Médios para Paciente {patient_name}:")
            for model_name, fold_metrics in patient_model_fold_metrics.items():
                 if fold_metrics:
                     avg_acc_pat = np.mean([m['acc'] for m in fold_metrics]); avg_f1_pat = np.mean([m['f1'] for m in fold_metrics]); avg_precision_pat = np.mean([m['precision'] for m in fold_metrics]); avg_recall_pat = np.mean([m['recall'] for m in fold_metrics])
                     print(f"    {model_name}: Acc={avg_acc_pat:.3f}, F1={avg_f1_pat:.3f}, P={avg_precision_pat:.3f}, R={avg_recall_pat:.3f}")
                     all_models_results[model_name]['acc'].append(avg_acc_pat)
                     all_models_results[model_name]['f1'].append(avg_f1_pat)
                     all_models_results[model_name]['precision'].append(avg_precision_pat)
                     all_models_results[model_name]['recall'].append(avg_recall_pat)
                 else: print(f"    {model_name}: Nenhum fold concluído.")

            patient_end_time = time.time(); print(f"  Tempo Paciente {patient_name}: {(patient_end_time - patient_start_time) / 60:.2f} min"); gc.collect()

        print("\n\n" + "="*40 + "\n" + "RESULTADO FINAL (MÉDIA GERAL POR MODELO)".center(40) + "\n" + "="*40)
        for model_name, results_dict in all_models_results.items():
            if results_dict['f1']: 
                avg_acc_all = np.mean(results_dict['acc']); std_acc_all = np.std(results_dict['acc'])
                avg_f1_all = np.mean(results_dict['f1']); std_f1_all = np.std(results_dict['f1'])
                avg_precision_all = np.mean(results_dict['precision']); std_precision_all = np.std(results_dict['precision'])
                avg_recall_all = np.mean(results_dict['recall']); std_recall_all = np.std(results_dict['recall'])
                print(f"\n--- Modelo: {model_name} ---")
                print(f"  Acurácia Média Geral:      {avg_acc_all:.3f} +/- {std_acc_all:.3f}")
                print(f"  F1 Score Médio Geral:      {avg_f1_all:.3f} +/- {std_f1_all:.3f}")
                print(f"  Precisão Média Geral:      {avg_precision_all:.3f} +/- {std_precision_all:.3f}")
                print(f"  Recall (Sens.) Médio Geral: {avg_recall_all:.3f} +/- {std_recall_all:.3f}")
            else:
                 print(f"\n--- Modelo: {model_name} ---"); print("  Nenhum resultado.")

    except Exception as e: print(f"\nERRO GERAL: {e}")
    finally:
        overall_end_time = time.time(); print(f"\nTempo total: {(overall_end_time - overall_start_time) / 60:.2f} min", flush=True)

if __name__ == '__main__':
    main()

Serviço do Google Drive conectado.
Caminho do dataset encontrado no Drive!

==================== Iniciando Paciente 1/1: chb01 ====================
    Download de temp_chb01_summary.txt (tentativa 1/3)... OK (1.24s).
  Fase 1a: Processando 7 arquivos COM crises...
    Download de temp_chb01_03.edf (tentativa 1/3)... OK (35.72s).
    chb01_03.edf: Extraindo features 1439 janelas (-1 cores)... OK (126.10s)
    1/7: chb01_03.edf processado em 164.68s. ETA: 16.5 min
    Download de temp_chb01_04.edf (tentativa 1/3)... OK (40.41s).
    chb01_04.edf: Extraindo features 1439 janelas (-1 cores)... OK (99.99s)
    2/7: chb01_04.edf processado em 143.21s. ETA: 12.8 min
    Download de temp_chb01_15.edf (tentativa 1/3)... OK (42.92s).
    chb01_15.edf: Extraindo features 1439 janelas (-1 cores)... OK (98.99s)
    3/7: chb01_15.edf processado em 144.38s. ETA: 10.1 min
    Download de temp_chb01_16.edf (tentativa 1/3)... OK (39.72s).
    chb01_16.edf: Extraindo features 1439 janelas (-1 cores)... 

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(


      Resultados KNN: Acc=0.985, F1=0.909, P=1.000, R=0.833
      Treinando DecisionTree... OK (0.07s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados DecisionTree: Acc=0.985, F1=0.909, P=1.000, R=0.833
    Fold 1 concluído em 3.18s

    --- FOLD 2/7 (chb01_04.edf_s1 fora) ---
      Treino: 1947 ([1770  177]), Teste: 143 ([130  13])
      Treinando RandomForest... OK (0.39s). Predição: 0.03s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados RandomForest: Acc=0.986, F1=0.917, P=1.000, R=0.846
      Treinando GradientBoosting... OK (2.36s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados GradientBoosting: Acc=0.993, F1=0.960, P=1.000, R=0.923
      Treinando SVM... OK (0.02s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados SVM: Acc=0.993, F1=0.960, P=1.000, R=0.923
      Treinando LogisticRegression... OK (0.01s). Predição: 0.00s. Aplicando suavização (j

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(


    Fold 2 concluído em 3.03s

    --- FOLD 3/7 (chb01_15.edf_s1 fora) ---
      Treino: 1892 ([1720  172]), Teste: 198 ([180  18])
      Treinando RandomForest... OK (0.29s). Predição: 0.02s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados RandomForest: Acc=0.929, F1=0.364, P=1.000, R=0.222
      Treinando GradientBoosting... OK (1.41s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados GradientBoosting: Acc=0.985, F1=0.909, P=1.000, R=0.833
      Treinando SVM... OK (0.02s). Predição: 0.01s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados SVM: Acc=0.955, F1=0.667, P=1.000, R=0.500
      Treinando LogisticRegression... OK (0.01s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados LogisticRegression: Acc=0.995, F1=0.971, P=1.000, R=0.944
      Treinando KNN... OK (0.00s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados KNN: Acc=0.924, F

...

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(


    Fold 5 concluído em 2.09s

    --- FOLD 6/7 (chb01_21.edf_s1 fora) ---
      Treino: 1661 ([1510  151]), Teste: 429 ([390  39])
      Treinando RandomForest... OK (0.30s). Predição: 0.03s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados RandomForest: Acc=0.993, F1=0.960, P=1.000, R=0.923
      Treinando GradientBoosting... OK (1.26s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados GradientBoosting: Acc=0.988, F1=0.932, P=1.000, R=0.872
      Treinando SVM... OK (0.02s). Predição: 0.01s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados SVM: Acc=0.991, F1=0.949, P=0.949, R=0.949
      Treinando LogisticRegression... OK (0.01s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados LogisticRegression: Acc=0.995, F1=0.974, P=1.000, R=0.949
      Treinando KNN... OK (0.00s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados KNN: Acc=0.993, F

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(


    Fold 6 concluído em 1.85s

    --- FOLD 7/7 (chb01_26.edf_s1 fora) ---
      Treino: 1617 ([1470  147]), Teste: 473 ([430  43])
      Treinando RandomForest... OK (0.27s). Predição: 0.03s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados RandomForest: Acc=0.994, F1=0.964, P=1.000, R=0.930
      Treinando GradientBoosting... OK (1.27s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados GradientBoosting: Acc=0.992, F1=0.951, P=1.000, R=0.907
      Treinando SVM... OK (0.02s). Predição: 0.01s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados SVM: Acc=0.996, F1=0.976, P=1.000, R=0.953
      Treinando LogisticRegression... OK (0.01s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados LogisticRegression: Acc=0.994, F1=0.964, P=1.000, R=0.930
      Treinando KNN... OK (0.00s). Predição: 0.00s. Aplicando suavização (janela=3, 5.0s)... OK (0.00s)
      Resultados KNN: Acc=0.992, F

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(


##### RESULTADOS

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

resultados_1_paciente = {
    'HDC Padrão': {
        'Acurácia': 0.937, 'F1-Score': 0.712, 'Precisão': 0.624, 'Recall': 0.845,
    },
    'HDC Multi-Pass': {
        'Acurácia': 0.958, 'F1-Score': 0.785, 'Precisão': 0.765, 'Recall': 0.826,
    },
    'HDC Multi-Cent': {
        'Acurácia': 0.863, 'F1-Score': 0.544, 'Precisão': 0.400, 'Recall': 0.876,
    },
    'HDC MCri': {
        'Acurácia': 0.863, 'F1-Score': 0.544, 'Precisão': 0.400, 'Recall': 0.876, 
    },
    'RandomForest': {
        'Acurácia': 0.981, 'F1-Score': 0.859, 'Precisão': 1.000, 'Recall': 0.796,
    },
    'GradientBoost': {
        'Acurácia': 0.990, 'F1-Score': 0.938, 'Precisão': 1.000, 'Recall': 0.886,
    },
    'SVM': {
        'Acurácia': 0.989, 'F1-Score': 0.926, 'Precisão': 0.993, 'Recall': 0.884,
    },
    'LogRegression': {
        'Acurácia': 0.995, 'F1-Score': 0.972, 'Precisão': 1.000, 'Recall': 0.945,
    },
    'KNN': {
        'Acurácia': 0.982, 'F1-Score': 0.856, 'Precisão': 1.000, 'Recall': 0.803,
    },
    'DecisionTree': {
        'Acurácia': 0.980, 'F1-Score': 0.851, 'Precisão': 1.000, 'Recall': 0.781,
    },
}

matrizes_confusao_1_paciente = {
    'HDC Padrão': np.array([[1810, 90], [36, 154]]),
    'HDC Multi-Pass': np.array([[1857, 43], [40, 150]]),
    'HDC Multi-Cent': np.array([[1649, 251], [25, 165]]),
    'HDC MCri': np.array([[1649, 251], [25, 165]]),
    'RandomForest': np.array([[1900, 0], [30, 160]]),
    'LogRegression': np.array([[1900, 0], [9, 181]]),
    'GradientBoost': np.array([[1900, 0], [21, 169]]),
    'SVM': np.array([[1893, 7], [22, 168]]),
    'KNN': np.array([[1900, 0], [37, 153]]),
    'DecisionTree': np.array([[1898, 2], [41, 149]]),
}

def plotar_barras_separadas_por_metrica(data_dict, run_name="01 Pacientes"):
    print(f"\nGerando gráficos de BARRAS VERTICAIS ({run_name})...")

    df_data = {}
    for model, metrics_dict in data_dict.items():
        df_data[model] = {m: v[0] if isinstance(v, tuple) else v for m, v in metrics_dict.items()}

    df = pd.DataFrame.from_dict(df_data, orient='index')

    metrics = ['Acurácia', 'F1-Score', 'Precisão', 'Recall']
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(metrics)))

    for metrica, color in zip(metrics, colors):

        df_sorted = df.sort_values(by=metrica, ascending=False)

        plt.figure(figsize=(12, 7))
        ax = plt.gca()

        bars = ax.bar(df_sorted.index, df_sorted[metrica], color=color, width=0.55)

        ax.set_title(f'Comparação de {metrica} entre Modelos\n({run_name})',
                     fontsize=16, pad=18, weight='bold')
        ax.set_ylabel('Pontuação da Métrica', fontsize=12)
        ax.set_xlabel('Modelos', fontsize=12)

        ax.set_ylim(0, 1.05)
        ax.set_xticks(range(len(df_sorted.index)))
        ax.set_xticklabels(df_sorted.index, rotation=45, ha='right')

        for bar in bars:
            y = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, y + 0.015, f"{y:.3f}",
                    ha='center', fontsize=10, weight='bold')

        plt.tight_layout()
        filename = f'01 {metrica}.png'
        plt.savefig(filename, dpi=300)
        print(f" -> Gráfico salvo como '{filename}'")
        plt.close()


def plotar_matrizes_confusao(cm_dict, output_dir='matrizes_confusao_1_paciente'):
    print(f"\nGerando Gráfico 2: Matrizes de Confusão...")

    if not isinstance(cm_dict, dict):
        raise ValueError("O parâmetro cm_dict deve ser um dicionário {'modelo': matriz_numpy}")

    if len(cm_dict) == 0:
        raise ValueError("O dicionário de matrizes está vazio!")

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for model_name, cm in cm_dict.items():

        if not isinstance(cm, np.ndarray):
            print(f"  -> Pulando '{model_name}' (matriz não é numpy array)")
            continue

        if cm.shape != (2, 2):
            print(f"  -> Pulando '{model_name}' — matriz não é 2x2 (é {cm.shape})")
            continue

        plt.figure(figsize=(6, 5))

        labels = np.array([[f"{cm[0,0]}", f"{cm[0,1]}"],
                           [f"{cm[1,0]}", f"{cm[1,1]}"]])

        sns.heatmap(
            cm,
            annot=labels,
            fmt="",
            cmap='Blues',
            cbar=False,
            annot_kws={"size": 14, "weight": "bold"},
            xticklabels=['Prev. Não Crise', 'Prev. Crise'],
            yticklabels=['Real Não Crise', 'Real Crise']
        )

        plt.title(f'Matriz de Confusão - {model_name}\n(Paciente chb01)', fontsize=15, pad=18)
        plt.ylabel('Classe Real', fontsize=12)
        plt.xlabel('Classe Prevista', fontsize=12)

        plt.tight_layout()

        filename = os.path.join(output_dir, f'MC_{model_name}.png')
        plt.savefig(filename, dpi=300)

        print(f"  -> Matriz '{model_name}' salva como '{filename}'")

        plt.close()


if __name__ == "__main__":
    print("Iniciando geração de gráficos e relatórios (1 Paciente)...")
    
    plotar_barras_separadas_por_metrica(resultados_1_paciente, run_name="1 Paciente")
        
    plotar_matrizes_confusao(matrizes_confusao_1_paciente, output_dir='matrizes_confusao_1_paciente')
    
    print("\nProcesso concluído! Verifique os arquivos .png.")

Iniciando geração de gráficos e relatórios (1 Paciente)...

Gerando gráficos de BARRAS VERTICAIS (1 Paciente)...
 -> Gráfico salvo como '01 Acurácia.png'
 -> Gráfico salvo como '01 F1-Score.png'
 -> Gráfico salvo como '01 Precisão.png'
 -> Gráfico salvo como '01 Recall.png'

Gerando Gráfico 2: Matrizes de Confusão...
  -> Matriz 'HDC Padrão' salva como 'matrizes_confusao_1_paciente/MC_HDC Padrão.png'
  -> Matriz 'HDC Multi-Pass' salva como 'matrizes_confusao_1_paciente/MC_HDC Multi-Pass.png'
  -> Matriz 'HDC Multi-Cent' salva como 'matrizes_confusao_1_paciente/MC_HDC Multi-Cent.png'
  -> Matriz 'HDC MCri' salva como 'matrizes_confusao_1_paciente/MC_HDC MCri.png'
  -> Matriz 'RandomForest' salva como 'matrizes_confusao_1_paciente/MC_RandomForest.png'
  -> Matriz 'LogRegression' salva como 'matrizes_confusao_1_paciente/MC_LogRegression.png'
  -> Matriz 'GradientBoost' salva como 'matrizes_confusao_1_paciente/MC_GradientBoost.png'
  -> Matriz 'SVM' salva como 'matrizes_confusao_1_paciente